# 03 Manuscript statistics

**Do Graph Representations Add Value Beyond Tabular Disease-History Features for Depression Prediction? A Controlled Comparison in UK Biobank.**

This notebook recomputes **every statistic that appears in the manuscript text and tables** from the aggregate result files written by `02_experiments.ipynb`, and prints each one as a table. Nothing is written to disk (see `WRITE_FILES` below).

**What it reads** (all located automatically under `DATA_DIR`):
`results_per_fold_*.csv`, `results_summary_*.csv`, `rcv_results_per_fold_*.csv` (per-fold metrics of the main run and of the two extra repeats), the `bootstrap_*.csv` tables (patient-level bootstrap, four classifiers) and, only for cross-checking, the `rcv_*.csv` tables written by the repeated-CV cell.

**How it is organised**

| Part | Content | Manuscript location |
|---|---|---|
| 1 | cohort, design and column counts | Methods, `tab:flow`, `tab:representations`, events per column |
| 2 | levels of every arm and classifier | `tab:levels`, AUPR paragraph, threshold metrics |
| 3 | paired contrasts of the main run (corrected resampled-*t* and bootstrap) | `tab:keycontrasts`, `tab:clf31`, `tab:answers` |
| 4 | classifier contrasts | `tab:clfcontrasts` |
| 5 | calibration and tuned-model selection | `tab:calibration`, `tab:calibration-other`, tuning paragraph |
| 6 | repeated cross-validation | `tab:paired-tuned`, `tab:paired-full`, appendix text |
| 7 | the 26 additional arms and the F41/F43 ablation | `tab:arms26`, `tab:nof`, findings list |
| 8 | **claims register**: every number quoted in the running text, its recomputed value and a status | Abstract, Results, Discussion, appendices |

**Status column of the claims register**: `OK` = the recomputed value rounds to the printed one; `CLOSE` = differs by one unit in the last printed digit (usually a rounding convention or a value stored with fewer digits); `DIFF` = a real difference to look at; `not recomputable` = needs patient-level data that are not among the files (e.g. the cohort flow, graph tie diagnostics).

**Inputs that are not in any result file** (cohort counts, number of tuned columns, graph tie shares) are typed in once in the `COHORT` and `STATIC` blocks below and are labelled as such wherever they are used.

In [ ]:
import os, re, math, ast, itertools, warnings
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import pandas as pd
from scipy import stats

try:                                   # inside Jupyter
    from IPython.display import display, Markdown
except Exception:                      # plain python: fall back to print
    def display(*objs, **kw):
        for o in objs:
            print(o)
    def Markdown(s):
        return s

# ============================== CONFIG ==============================
DATA_DIR    = 'results'    # folder with the aggregate CSV files written by 02_experiments.ipynb (searched recursively; files may carry a hash prefix 'xxxx-')
WRITE_FILES = False        # True -> additionally write all tables to ONE Excel workbook (EXPORT_PATH). False -> no file is created.
EXPORT_PATH = os.path.join(DATA_DIR, 'manuscript_statistics.xlsx')
BUILD_TAG   = 'exact_nested_v1_mltesplit_onsetscaled'

# ---- cohort numbers that come from the UK Biobank cohort itself (copied from the manuscript, NOT recomputable from the result files)
COHORT = dict(n_ukb=501_936, n_after_gqc=407_606, n_townsend_nmr=401_695, n_baseline_date=75_160,
              n_analysed=68_618, n_events=3_292, n_died_no_f32=7_062)
# ---- other static numbers quoted in the manuscript that need patient-level data / the graph builders (shown for completeness)
STATIC = dict(share_no_history=0.126, n_vocab=207, n_vocab_used=203, n_raw_features=62, mlte_dim=256, k=50,
              graphdiag={'OAO, 500-candidate pool (main)': (0.465, 0.455, 0.475), 'OAO, 51-candidate pool': (0.612, 0.609, 0.615),
                         'MDBT': (0.466, 0.464, 0.468), 'DCEG': (0.266, 0.264, 0.269), 'MLTE': (0.037, 0.022, 0.057),
                         'OAO without F41/F43, 500-candidate pool': (0.469, 0.459, 0.477), 'MDBT without F41/F43': (0.465, 0.461, 0.469),
                         'DCEG without F41/F43': (0.267, 0.265, 0.269), 'Feature-kNN': (0.001, 0.001, 0.001)})
N_FOLDS = 10

pd.set_option('display.max_columns', None); pd.set_option('display.max_rows', 600)
pd.set_option('display.width', 250);        pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda v: f'{v:.6g}')

TABLES = {}                     # name -> DataFrame (everything that is displayed is also kept here)
def show(name, df, title=None, index=True):
    """Display a table with a heading and keep it in TABLES."""
    TABLES[name] = df
    display(Markdown(f'**{title or name}**  \n`{name}` — {len(df)} rows × {df.shape[1]} columns'))
    display(df if index else df.reset_index(drop=True))
    return df

# ---------------------------------------------------------------- file finder
def _roots():
    seen = []
    for r in [DATA_DIR, '.']:
        r = os.path.abspath(r)
        if os.path.isdir(r) and r not in seen:
            seen.append(r)
    return seen

def find(name, required=True):
    """Return the path of `name` (exact name, or with a '<hash>-' prefix) searching DATA_DIR first."""
    for root in _roots():
        for dp, dn, fn in os.walk(root):
            if dp[len(root):].count(os.sep) > 3:
                dn[:] = []; continue
            for f in fn:
                if f == name or f.endswith('-' + name):
                    return os.path.join(dp, f)
    if required:
        raise FileNotFoundError(f'{name} not found under {_roots()}; set DATA_DIR')
    return None

def read(name, **kw):
    return pd.read_csv(find(name), **kw)

print('searching in:', _roots())

## Helpers
Half-up rounding (as a typesetter would), the corrected resampled-*t* test of Nadeau & Bengio, Holm/BH adjustment, manuscript-style number formatting, and the claims register.

In [ ]:
def rnd(x, d):
    """Round half up (652.5 -> 653), on the decimal representation of the float."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return float(Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP))

def sg(x, d=4):          # signed number as in the tables:  +0.0234 / -0.0005
    return f'{x:+.{d}f}'
def ci(lo, hi, d=3):     # [+0.016, +0.031]
    return f'[{lo:+.{d}f}, {hi:+.{d}f}]'
def pstr(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def split_sizes(n, k=N_FOLDS):
    """Reproduces KFold(k, shuffle) + train_test_split(test_size=1/9) sizes of the experiments notebook."""
    rows = []
    for f in range(k):
        n_test = n // k + (1 if f < n % k else 0)
        n_tv = n - n_test
        n_val = math.ceil((1.0 / 9.0) * n_tv)
        rows.append(dict(fold=f, n_train=n_tv - n_val, n_val=n_val, n_test=n_test))
    d = pd.DataFrame(rows).set_index('fold')
    d['ratio_test_over_train'] = d.n_test / d.n_train
    return d
SPLITS = split_sizes(COHORT['n_analysed'])
RATIO = float(SPLITS.ratio_test_over_train.mean())         # n_test / n_train used in the corrected resampled t

def corrected_t(d, ratio=None, alpha=0.05):
    """Corrected resampled t (Nadeau & Bengio 2003; Bouckaert & Frank 2004 for repeated CV):
    var = (1/J + n_test/n_train) s^2, df = J-1.  Returns mean, lo, hi, two-sided p."""
    ratio = RATIO if ratio is None else ratio
    d = np.asarray(d, float); J = len(d); m = d.mean(); s2 = d.var(ddof=1)
    se = math.sqrt((1.0 / J + ratio) * s2)
    tc = stats.t.ppf(1 - alpha / 2, J - 1)
    p = 2 * stats.t.sf(abs(m) / se, J - 1) if se > 0 else 1.0
    return m, m - tc * se, m + tc * se, p

def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def bh(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 1.0
    for r, i in enumerate(np.argsort(p)[::-1]):
        run = min(run, p[i] * m / (m - r)); adj[i] = run
    return adj

# ---------------------------------------------------------------- claims register
CLAIMS = []
_RANK = {'OK': 0, 'CLOSE': 1, 'DIFF': 2}
def _num(x, dec, signed):
    if dec is None or dec <= 0:
        return f'{x:+,.0f}' if signed else f'{x:,.0f}'
    return f'{x:+.{dec}f}' if signed else f'{x:.{dec}f}'
def _status(ms, val, dec):
    shown = rnd(val, dec); tol = 10.0 ** (-dec) if dec > 0 else 10.0 ** (-dec)
    diff = abs(shown - ms)
    return shown, ('OK' if diff < tol * 1e-6 else ('CLOSE' if diff <= tol * 1.0001 else 'DIFF'))

def claim_range(section, statement, ms_lo, ms_hi, vals, dec, signed=False, note=''):
    """A range (or single value if ms_lo == ms_hi) quoted in the text.  `vals` = the recomputed quantities it summarises."""
    vals = [float(v) for v in vals]
    lo, hi = min(vals), max(vals)
    (s_lo, st_lo), (s_hi, st_hi) = _status(ms_lo, lo, dec), _status(ms_hi, hi, dec)
    st = st_lo if _RANK[st_lo] >= _RANK[st_hi] else st_hi
    one = (ms_lo == ms_hi)
    CLAIMS.append(dict(section=section, statement=statement,
                       manuscript=_num(ms_lo, dec, signed) if one else f'{_num(ms_lo, dec, signed)} to {_num(ms_hi, dec, signed)}',
                       recomputed=_num(s_lo, dec, signed) if (s_lo == s_hi) else f'{_num(s_lo, dec, signed)} to {_num(s_hi, dec, signed)}',
                       exact=(f'{lo:.6g}' if lo == hi else f'{lo:.6g} .. {hi:.6g}'), status=st, note=note))

def claim(section, statement, ms, val, dec, signed=False, note=''):
    """A single number quoted in the text.  val=None -> cannot be recomputed from the files."""
    if val is None:
        CLAIMS.append(dict(section=section, statement=statement, manuscript=_num(ms, dec, signed), recomputed='', exact='',
                           status='not recomputable', note=note)); return
    claim_range(section, statement, ms, ms, [val], dec, signed, note)

def claim_true(section, statement, cond, detail='', note=''):
    CLAIMS.append(dict(section=section, statement=statement, manuscript='true', recomputed=str(bool(cond)).lower(),
                       exact=detail, status='OK' if cond else 'DIFF', note=note))

def claim_na(section, statement, ms, note=''):
    CLAIMS.append(dict(section=section, statement=statement, manuscript=str(ms), recomputed='', exact='',
                       status='not recomputable', note=note))
print('ratio n_test/n_train =', round(RATIO, 6))

## Load the result files

In [ ]:
PF  = read(f'results_per_fold_{BUILD_TAG}.csv')              # main run: arm x classifier x fold
RCV = read(f'rcv_results_per_fold_{BUILD_TAG}.csv')          # 3 repeats x 10 folds, logistic regressions only
SUM = read(f'results_summary_{BUILD_TAG}.csv', header=[0, 1], index_col=[0, 1])    # mean/std per arm x classifier (written by the experiments notebook)
SUM.index.names = ['model', 'arm']

CLF = {'tLR': 'LogisticRegressionTuned', 'dLR': 'LogisticRegression', 'dCB': 'CatBoost', 'tCB': 'CatBoostTuned'}
CLF_NAME = {'LogisticRegressionTuned': 'Tuned LR (primary)', 'LogisticRegression': 'Default L2 LR',
            'CatBoost': 'Default CatBoost', 'CatBoostTuned': 'Tuned CatBoost'}
TLR, DLR, DCB, TCB = CLF['tLR'], CLF['dLR'], CLF['dCB'], CLF['tCB']

# manuscript label -> arm key (the 14 main representations)
LABEL = {'Raw': 'raw', 'Raw + onset': 'raw_onset', 'OAO': 'oaopool500_50', 'MDBT': 'mdbt_50', 'DCEG': 'dceg_50', 'MLTE': 'mlte_50',
         'Random graph': 'random_50', 'Age-and-sex graph': 'agesex_50', 'Shuffled history (DCEG)': 'shufrow_dceg_50',
         'OAO + onset': 'oaopool500_50_onset', 'MDBT + onset': 'mdbt_50_onset', 'DCEG + onset': 'dceg_50_onset',
         'MLTE + onset': 'mlte_50_onset', 'Raw + MLTE vector': 'raw_mltevec'}
ARM2LABEL = {v: k for k, v in LABEL.items()}
MAIN14 = list(LABEL.values())
GRAPHS4 = ['oaopool500_50', 'mdbt_50', 'dceg_50', 'mlte_50']
HAND3   = ['oaopool500_50', 'mdbt_50', 'dceg_50']
CTRL3   = ['random_50', 'agesex_50', 'shufrow_dceg_50']
COMB4   = ['oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset', 'mlte_50_onset']

def pair(s):
    a, b = s.split(' vs. ')
    return LABEL[a.strip()], LABEL[b.strip()]
def pair_label(a, b):
    return f'{ARM2LABEL.get(a, a)} vs. {ARM2LABEL.get(b, b)}'

def wide(df, model, metric):
    """fold (or repeat x fold) x arm matrix of one metric for one classifier"""
    idx = [c for c in ('repeat', 'fold') if c in df.columns]
    return df[df.model == model].pivot_table(index=idx, columns='arm', values=metric)

W = {(m, met): wide(PF, m, met) for m in CLF.values() for met in ('auc', 'aupr', 'accuracy', 'recall', 'brier', 'brier_cal',
                                                                 'calibration_intercept', 'calibration_slope', 'ece',
                                                                 'calibration_intercept_cal', 'calibration_slope_cal', 'ece_cal')}
WR = {(m, met): wide(RCV, m, met) for m in (TLR, DLR) for met in ('auc', 'aupr')}

info = pd.DataFrame({'rows': PF.groupby('model').size(), 'arms': PF.groupby('model').arm.nunique(), 'folds': PF.groupby('model').fold.nunique()})
display(Markdown('**Loaded: main run per-fold file**'));  display(info)
display(Markdown(f'**Loaded: repeated CV per-fold file** — {len(RCV)} rows, repeats {sorted(RCV.repeat.unique().tolist())}, arms {RCV.arm.nunique()}, classifiers {sorted(RCV.model.unique())}'))
# the repeated-CV repeat 0 must equal the main run
chk = max(abs(WR[(m, 'auc')].loc[0] - W[(m, 'auc')][WR[(m, 'auc')].columns]).abs().max().max() for m in (TLR, DLR))
print('max |repeat 0 - main run| in AUROC:', chk)
assert chk < 1e-12

# Part 1 — Cohort, design and column counts
Counts of the cohort cannot be recomputed from the result files (they come from the UK Biobank extraction); they are typed in `COHORT` and the derived percentages are recomputed.

In [ ]:
n, ev = COHORT['n_analysed'], COHORT['n_events']
flow = pd.DataFrame({'participants': [COHORT['n_ukb'], COHORT['n_after_gqc'], COHORT['n_townsend_nmr'], COHORT['n_baseline_date'], n]},
                    index=['UK Biobank participants', 'After genetic quality control', 'With Townsend index and complete NMR measures',
                           'Baseline on or before 31 March 2008', 'After excluding F32 at or before baseline'])
flow['removed_vs_previous_step'] = -flow.participants.diff()
flow['share_removed_%'] = 100 * flow.removed_vs_previous_step / flow.participants.shift(1)
flow['retained_of_total_%'] = 100 * flow.participants / flow.participants.iloc[0]
show('flow', flow, 'tab:flow — participant flow (counts typed in from the manuscript; shares recomputed)')

outcome = pd.DataFrame({'n': [ev, n - ev, COHORT['n_died_no_f32']], '% of analysed': [100 * ev / n, 100 * (n - ev) / n, 100 * COHORT['n_died_no_f32'] / n]},
                       index=['Incident F32 within 15 years', 'No incident F32', 'Died within 15 years without F32 (non-events)'])
show('outcome', outcome, 'Outcome counts and event rate')
rate = ev / n
derived = pd.Series({'event rate': rate, 'no-skill AUPR (= event rate)': rate, 'all-negative accuracy (1 - rate)': 1 - rate,
                     'Brier score of a constant prediction  p(1-p)': rate * (1 - rate),
                     'validation events per fold  (n_val x rate)': SPLITS.n_val.mean() * rate,
                     'data / validation / test split (mean over folds): train': SPLITS.n_train.mean(),
                     'validation': SPLITS.n_val.mean(), 'test': SPLITS.n_test.mean()}, name='value')
show('derived_constants', derived.to_frame(), 'Quantities derived from the cohort counts')
show('split_sizes', SPLITS, 'Split sizes per fold (KFold 10 + validation = 1/9 of the remainder); n_test/n_train enters the corrected-t')

claim('Methods', 'events / total = incident F32 rate (%)', 4.8, 100 * rate, 1)
claim('Results/flow', 'incident F32 rate in tab:flow (%)', 4.80, 100 * rate, 2)
claim('Results', 'died within 15 years without F32 (%)', 10.3, 100 * COHORT['n_died_no_f32'] / n, 1)
claim('Results/flow', 'died without F32 in tab:flow (%)', 10.29, 100 * COHORT['n_died_no_f32'] / n, 2)
claim('Methods', 'no-skill AUPR', 0.048, rate, 3)
claim('Appendix C', 'all-negative accuracy', 0.9520, 1 - rate, 4)
claim('Appendix C', 'Brier score of a constant prediction', 0.0457, rate * (1 - rate), 4)
claim('Appendix C', 'validation events per fold', 330, SPLITS.n_val.mean() * rate, -1)
claim('Methods', 'training participants per fold (about 54,900)', 54900, SPLITS.n_train.mean(), -2)
claim('Methods', 'validation participants per fold (about 6,860)', 6860, SPLITS.n_val.mean(), -1)
claim('Methods', 'test participants per fold (about 6,860)', 6860, SPLITS.n_test.mean(), -1)
claim('Discussion', 'date restriction keeps 75,160 of 401,695 eligible participants', 75160, COHORT['n_baseline_date'], 0)

### Columns per representation and events per candidate column
`n_features` comes straight from the per-fold file; the manuscript quotes about 53, 7, 18, 6 and 10 events per column for raw, raw + onset, graph, combined and raw + embedding.

In [ ]:
cols = (PF.groupby('arm').n_features.agg(['min', 'max']).rename(columns={'min': 'n_columns', 'max': 'n_columns_max'}))
assert (cols.n_columns == cols.n_columns_max).all()
tab1_cols = {'raw': 62, 'raw_onset': 468, 'oaopool500_50': 186, 'mdbt_50': 186, 'dceg_50': 186, 'mlte_50': 186, 'random_50': 186,
             'agesex_50': 186, 'shufrow_dceg_50': 186, 'oaopool500_50_onset': 592, 'mdbt_50_onset': 592, 'dceg_50_onset': 592,
             'mlte_50_onset': 592, 'raw_mltevec': 318}
rep = pd.DataFrame({'representation': [ARM2LABEL[a] for a in MAIN14], 'columns (data)': [int(cols.loc[a, 'n_columns']) for a in MAIN14],
                    'columns (tab:representations)': [tab1_cols[a] for a in MAIN14]}, index=MAIN14)
rep['match'] = rep['columns (data)'] == rep['columns (tab:representations)']
rep['events per column'] = ev / rep['columns (data)']
show('representations', rep, 'tab:representations — columns per main representation and events per column')

epv = {'raw': 53, 'raw_onset': 7, 'oaopool500_50': 18, 'oaopool500_50_onset': 6, 'raw_mltevec': 10}
for a, v in epv.items():
    claim('Methods', f'events per candidate column, {ARM2LABEL[a]}', v, ev / cols.loc[a, 'n_columns'], 0)
claim_true('Methods', 'column counts of the 14 main arms agree with tab:representations', rep.match.all())
claim('Methods', 'onset-age block = 2 x 203 columns; raw + onset = 62 + 406', 468, 62 + 2 * STATIC['n_vocab_used'], 0)
claim('Methods', 'graph = 3 x 62 columns', 186, 3 * 62, 0)
claim('Methods', 'combined = 186 + 406 columns', 592, 186 + 406, 0)
claim('Methods', 'raw + MLTE vector = 62 + 256 columns', 318, 62 + STATIC['mlte_dim'], 0)

# Part 2 — Levels: AUROC / AUPR per representation and classifier
Fold means come from the main-run per-fold file; the 95% intervals are the patient-level bootstrap intervals stored in `bootstrap_arms*.csv` (test-sampling variability only; they cannot be recomputed because the pooled test predictions are not among the files).

In [ ]:
BA = {TLR: read('bootstrap_arms.csv', index_col=0), DLR: read('bootstrap_arms_logisticregression.csv', index_col=0),
      DCB: read('bootstrap_arms_catboost.csv', index_col=0), TCB: read('bootstrap_arms_catboosttuned.csv', index_col=0)}
def mean_of(model, met, arms):
    return W[(model, met)][arms].mean()

lv = pd.DataFrame(index=[ARM2LABEL[a] for a in MAIN14])
for m, nm in ((TLR, 'Tuned LR'), (DLR, 'Default LR'), (TCB, 'Tuned CB'), (DCB, 'Default CB')):
    lv[f'{nm} AUROC'] = mean_of(m, 'auc', MAIN14).values
lv['Tuned LR bootstrap lo'] = BA[TLR].loc[MAIN14, 'AUROC_lo'].values
lv['Tuned LR bootstrap hi'] = BA[TLR].loc[MAIN14, 'AUROC_hi'].values
lv['Tuned LR interval (as printed)'] = [f'{rnd(a,3):.3f} ({rnd(l,3):.3f}-{rnd(h,3):.3f})' for a, l, h in zip(lv['Tuned LR AUROC'], lv['Tuned LR bootstrap lo'], lv['Tuned LR bootstrap hi'])]
show('levels', lv.round(4), 'tab:levels — AUROC, mean over ten folds (all four classifiers) + bootstrap interval (tuned LR)')

# tab:levels cross-check against the printed table (values are 3-decimals)
LEVELS_MS = {  # tuned LR, default LR, tuned CB, as printed in tab:levels
 'Raw': (.655, .655, .658), 'Raw + onset': (.679, .664, .683), 'OAO': (.668, .668, .675), 'MDBT': (.674, .672, .676),
 'DCEG': (.671, .667, .676), 'MLTE': (.678, .678, .680), 'Random graph': (.655, .649, .653), 'Age-and-sex graph': (.655, .647, .655),
 'Shuffled history (DCEG)': (.655, .649, .654), 'OAO + onset': (.679, .662, .684), 'MDBT + onset': (.681, .662, .681),
 'DCEG + onset': (.681, .662, .682), 'MLTE + onset': (.684, .665, .683), 'Raw + MLTE vector': (.682, .676, .683)}
for lab, (a, b, c) in LEVELS_MS.items():
    claim('tab:levels', f'{lab}: tuned LR AUROC', a, lv.loc[lab, 'Tuned LR AUROC'], 3)
    claim('tab:levels', f'{lab}: default L2 LR AUROC', b, lv.loc[lab, 'Default LR AUROC'], 3)
    claim('tab:levels', f'{lab}: tuned CatBoost AUROC', c, lv.loc[lab, 'Tuned CB AUROC'], 3)
BOOT_MS = {'Raw': (.646, .664), 'Raw + onset': (.669, .688), 'OAO': (.659, .678), 'MDBT': (.665, .683), 'DCEG': (.662, .681), 'MLTE': (.669, .687),
           'Random graph': (.645, .664), 'Age-and-sex graph': (.645, .664), 'Shuffled history (DCEG)': (.646, .664), 'OAO + onset': (.669, .688),
           'MDBT + onset': (.671, .690), 'DCEG + onset': (.672, .690), 'MLTE + onset': (.675, .693), 'Raw + MLTE vector': (.672, .692)}
for lab, (lo, hi) in BOOT_MS.items():
    claim('tab:levels', f'{lab}: bootstrap lower limit', lo, lv.loc[lab, 'Tuned LR bootstrap lo'], 3)
    claim('tab:levels', f'{lab}: bootstrap upper limit', hi, lv.loc[lab, 'Tuned LR bootstrap hi'], 3)

# ranges quoted in the Results paragraph
A = lv['Tuned LR AUROC']
claim('Results', 'raw AUROC (tuned LR)', .655, A['Raw'], 3)
claim('Results', 'raw + onset AUROC', .679, A['Raw + onset'], 3)
claim_range('Results', 'handcrafted graphs AUROC range', .668, .674, A[['OAO', 'MDBT', 'DCEG']], 3)
claim('Results', 'MLTE graph AUROC', .678, A['MLTE'], 3)
claim_range('Results', 'combined representations AUROC range', .679, .684, A[['OAO + onset', 'MDBT + onset', 'DCEG + onset', 'MLTE + onset']], 3)

# AUPR table (all classifiers) and the AUPR paragraph
pr = pd.DataFrame(index=[ARM2LABEL[a] for a in MAIN14])
for m, nm in ((TLR, 'Tuned LR'), (DLR, 'Default LR'), (TCB, 'Tuned CB'), (DCB, 'Default CB')):
    pr[f'{nm} AUPR'] = mean_of(m, 'aupr', MAIN14).values
pr['Tuned LR bootstrap lo'] = BA[TLR].loc[MAIN14, 'AUPR_lo'].values
pr['Tuned LR bootstrap hi'] = BA[TLR].loc[MAIN14, 'AUPR_hi'].values
show('aupr_levels', pr.round(4), 'AUPR per representation (appendix paragraph "AUPR"; no-skill = event rate 0.048)')
P = pr['Tuned LR AUPR']
claim('App. AUPR', 'tuned LR raw', .095, P['Raw'], 3); claim('App. AUPR', 'tuned LR raw + onset', .115, P['Raw + onset'], 3)
claim('App. AUPR', 'OAO', .106, P['OAO'], 3); claim('App. AUPR', 'MDBT', .109, P['MDBT'], 3)
claim('App. AUPR', 'DCEG', .110, P['DCEG'], 3); claim('App. AUPR', 'MLTE', .112, P['MLTE'], 3)
claim_range('App. AUPR', 'control graphs', .095, .096, P[['Random graph', 'Age-and-sex graph', 'Shuffled history (DCEG)']], 3)
claim_range('App. AUPR', 'combined', .115, .118, P[['OAO + onset', 'MDBT + onset', 'DCEG + onset', 'MLTE + onset']], 3)
claim('App. AUPR', 'MLTE + onset', .118, P['MLTE + onset'], 3); claim('App. AUPR', 'raw + MLTE vector', .115, P['Raw + MLTE vector'], 3)
claim('App. AUPR', 'tuned CatBoost raw', .097, pr.loc['Raw', 'Tuned CB AUPR'], 3)
claim('App. AUPR', 'tuned CatBoost raw + onset', .110, pr.loc['Raw + onset', 'Tuned CB AUPR'], 3)

# threshold metrics
th = []
for m in CLF.values():
    d = PF[PF.model == m]
    g = d.groupby('arm')
    th.append(dict(classifier=CLF_NAME[m], arms=g.ngroups, acc_min=g.accuracy.mean().min(), acc_max=g.accuracy.mean().max(),
                   max_abs_dev_from_all_negative=(g.accuracy.mean() - (1 - rate)).abs().max(), max_recall_any_fold=d.recall.max(),
                   max_f1_any_fold=d.f1.max()))
th = pd.DataFrame(th).set_index('classifier')
show('threshold_metrics', th, 'Threshold metrics at the 0.5 cut-off (accuracy of every classifier/arm vs the all-negative rate)')
claim_range('App. threshold', 'mean accuracy of every classifier and arm', .9515, .9521, [th.acc_min.min(), th.acc_max.max()], 4)
claim('App. threshold', 'accuracy within 0.0005 of the all-negative rate (max deviation)', .0005, th.max_abs_dev_from_all_negative.max(), 4,
      note='"at or within 0.0005" -> the maximum must not exceed it')
claim('App. threshold', 'recall never exceeded 0.017 in any fold', .017, th.max_recall_any_fold.max(), 3)

# calibration tables
def calib_table(model, cols, fmt):
    rows = []
    for a in MAIN14:
        d = PF[(PF.model == model) & (PF.arm == a)]
        rows.append(pd.Series({k: d[v].mean() for k, v in cols.items()}, name=ARM2LABEL[a]))
    return pd.DataFrame(rows)
cal_dlr = calib_table(DLR, {'Brier': 'brier', 'Intercept': 'calibration_intercept', 'Slope': 'calibration_slope', 'ECE (eq-width)': 'ece',
                            'Brier (recal)': 'brier_cal', 'Intercept (recal)': 'calibration_intercept_cal', 'Slope (recal)': 'calibration_slope_cal',
                            'ECE (eq-freq, recal)': 'ece_cal'}, None)
show('calibration', cal_dlr.round(4), 'tab:calibration — default L2 logistic regression, arm means over ten folds')
cal_o = pd.concat({CLF_NAME[m]: calib_table(m, {'Brier': 'brier', 'Slope': 'calibration_slope', 'Brier (recal)': 'brier_cal', 'Slope (recal)': 'calibration_slope_cal'}, None)
                   for m in (TLR, TCB)}, axis=1)
show('calibration_other', cal_o.round(4), 'tab:calibration-other — tuned LR (primary) and tuned CatBoost')

CAL_DLR_MS = {  # Brier, int, slope, ECE, Brier_cal, int_cal, slope_cal, ECE_cal  (first and last rows are enough to catch a column shift; whole table checked below)
 'Raw': (.0449, -.20, .93, .0033, .0449, .02, 1.01, .0067), 'Raw + onset': (.0451, -.75, .73, .0061, .0447, .13, 1.05, .0071),
 'OAO': (.0447, -.30, .89, .0034, .0447, .04, 1.02, .0059), 'MDBT': (.0446, -.36, .87, .0039, .0446, .11, 1.04, .0070),
 'DCEG': (.0446, -.34, .88, .0040, .0446, .08, 1.03, .0067), 'MLTE': (.0445, -.30, .89, .0042, .0445, .10, 1.04, .0063),
 'Random graph': (.0450, -.37, .87, .0038, .0450, .05, 1.02, .0067), 'Age-and-sex graph': (.0450, -.45, .84, .0040, .0450, -.00, 1.00, .0063),
 'Shuffled history (DCEG)': (.0450, -.37, .87, .0035, .0450, .02, 1.01, .0065), 'OAO + onset': (.0451, -.82, .71, .0071, .0448, .11, 1.04, .0073),
 'MDBT + onset': (.0452, -.85, .70, .0073, .0448, .17, 1.06, .0056), 'DCEG + onset': (.0451, -.83, .70, .0072, .0448, .14, 1.05, .0073),
 'MLTE + onset': (.0451, -.81, .71, .0071, .0447, .15, 1.05, .0065), 'Raw + MLTE vector': (.0446, -.54, .81, .0054, .0445, .08, 1.03, .0069)}
decs = (4, 2, 2, 4, 4, 2, 2, 4)
for lab, ms in CAL_DLR_MS.items():
    for c, v, dd in zip(cal_dlr.columns, ms, decs):
        claim('tab:calibration', f'{lab}: {c}', v, cal_dlr.loc[lab, c], dd, signed=(dd == 2 and 'ntercept' in c))
CAL_O_MS = {'Raw': (.0449, .97, .0449, 1.02, .0449, 1.11, .0449, 1.00), 'Raw + onset': (.0445, .97, .0445, 1.04, .0446, 1.14, .0446, 1.02),
 'OAO': (.0447, .95, .0447, 1.03, .0447, 1.13, .0447, 1.02), 'MDBT': (.0446, .99, .0446, 1.04, .0446, 1.11, .0446, 1.04),
 'DCEG': (.0446, .99, .0446, 1.04, .0446, 1.10, .0446, 1.02), 'MLTE': (.0445, .97, .0445, 1.03, .0445, 1.11, .0445, 1.03),
 'Random graph': (.0449, .99, .0449, 1.01, .0449, 1.20, .0449, 1.01), 'Age-and-sex graph': (.0449, .96, .0449, 1.01, .0449, 1.14, .0449, 1.00),
 'Shuffled history (DCEG)': (.0449, 1.00, .0449, 1.01, .0449, 1.16, .0449, 1.00), 'OAO + onset': (.0445, .97, .0445, 1.04, .0446, 1.16, .0446, 1.03),
 'MDBT + onset': (.0445, .97, .0445, 1.05, .0446, 1.14, .0446, 1.04), 'DCEG + onset': (.0444, .97, .0444, 1.04, .0445, 1.14, .0445, 1.02),
 'MLTE + onset': (.0444, .97, .0444, 1.03, .0445, 1.11, .0445, 1.04), 'Raw + MLTE vector': (.0444, 1.02, .0444, 1.04, .0445, 1.09, .0445, 1.04)}
decs2 = (4, 2, 4, 2, 4, 2, 4, 2)
for lab, ms in CAL_O_MS.items():
    for c, v, dd in zip(cal_o.columns, ms, decs2):
        claim('tab:calibration-other', f'{lab}: {c[0]} / {c[1]}', v, cal_o.loc[lab, c], dd)

# tuned model selection (tuned LR)
tl = PF[PF.model == TLR].copy()
tl['main'] = tl.arm.isin(MAIN14)
sel = pd.DataFrame({'fits': [tl.main.sum(), len(tl)], 'share L1 (%)': [100 * (tl[tl.main].tuned_penalty == 'l1').mean(), 100 * (tl.tuned_penalty == 'l1').mean()],
                    'median C': [tl[tl.main].tuned_C.median(), tl.tuned_C.median()]}, index=['14 main arms', 'all arms (incl. sensitivity)'])
show('tuning_selection', sel, 'Tuned LR: penalty and C selected in the inner validation')
freq = tl[tl.main].groupby(['tuned_penalty', 'tuned_C']).size().sort_values(ascending=False).rename('fits (14 main arms)').to_frame()
show('tuning_choices', freq, 'Most frequent (penalty, C) choices among the 140 main-arm fits')
claim('App. tuning', 'fits of the 14 main representations', 140, tl.main.sum(), 0)
claim('App. tuning', 'all fits including sensitivity arms', 410, len(tl), 0)
claim('App. tuning', 'L1 share, 14 main (%)', 81, sel.iloc[0, 1], 0); claim('App. tuning', 'L1 share, all (%)', 80, sel.iloc[1, 1], 0)
claim('App. tuning', 'median C', 0.10, sel.iloc[0, 2], 2)
claim('App. tuning', 'L1, C=0.1 fits', 80, freq.loc[('l1', 0.1)].iloc[0] if ('l1', 0.1) in freq.index else 0, 0)
claim('App. tuning', 'L1, C=0.03 fits', 17, freq.loc[('l1', 0.03)].iloc[0] if ('l1', 0.03) in freq.index else 0, 0)
claim('App. tuning', 'L1, C=0.3 fits', 16, freq.loc[('l1', 0.3)].iloc[0] if ('l1', 0.3) in freq.index else 0, 0)

## Values printed in the manuscript tables (typed in for the comparison)
These literals are transcribed from the LaTeX tables so that every printed cell can be compared with its recomputed value. A negative Holm p in `MS_NOF` stands for the printed "<p".

In [ ]:
MS_KEY = [('Raw + onset vs. Raw', 0.0234, 0.016, 0.031, 0.018, 0.028, 0.0093, 0.0245), ('OAO vs. Raw', 0.0132, 0.006, 0.021, 0.009, 0.017, 0.0129, 0.0165), ('MDBT vs. Raw', 0.0191, 0.012, 0.026, 0.014, 0.024, 0.0169, 0.0179), ('DCEG vs. Raw', 0.0163, 0.01, 0.023, 0.012, 0.021, 0.0127, 0.0176), ('MLTE vs. Raw', 0.0226, 0.015, 0.031, 0.017, 0.028, 0.0228, 0.0224), ('Random graph vs. Raw', -0.0005, -0.001, 0.0, -0.001, 0.0, -0.0058, -0.0055), ('Age-and-sex graph vs. Raw', -0.0005, -0.002, 0.001, -0.001, 0.001, -0.0077, -0.0033), ('Shuffled history (DCEG) vs. Raw', 0.0002, -0.001, 0.001, -0.001, 0.001, -0.0052, -0.004), ('OAO vs. Random graph', 0.0137, 0.006, 0.021, 0.009, 0.018, 0.0187, 0.022), ('MDBT vs. Random graph', 0.0196, 0.012, 0.027, 0.014, 0.024, 0.0228, 0.0234), ('DCEG vs. Random graph', 0.0168, 0.01, 0.024, 0.012, 0.021, 0.0185, 0.0231), ('MLTE vs. Random graph', 0.0231, 0.015, 0.031, 0.017, 0.029, 0.0287, 0.0279), ('DCEG vs. Shuffled history (DCEG)', 0.0161, 0.009, 0.023, 0.011, 0.021, 0.0179, 0.0216), ('MLTE vs. Shuffled history (DCEG)', 0.0224, 0.014, 0.031, 0.017, 0.028, 0.0281, 0.0265), ('Raw + onset vs. OAO', 0.0102, -0.001, 0.021, 0.005, 0.015, -0.0036, 0.008), ('Raw + onset vs. MDBT', 0.0043, -0.001, 0.01, -0.0, 0.009, -0.0076, 0.0066), ('Raw + onset vs. DCEG', 0.0071, 0.002, 0.012, 0.002, 0.012, -0.0034, 0.0069), ('Raw + onset vs. MLTE', 0.0008, -0.006, 0.008, -0.004, 0.005, -0.0135, 0.0021), ('OAO + onset vs. Raw + onset', 0.0002, -0.002, 0.002, -0.001, 0.001, -0.0016, 0.0014), ('MDBT + onset vs. Raw + onset', 0.0025, -0.002, 0.007, 0.0, 0.005, -0.0023, -0.0012), ('DCEG + onset vs. Raw + onset', 0.0024, -0.0, 0.005, 0.0, 0.005, -0.0024, -0.0003), ('MLTE + onset vs. Raw + onset', 0.0054, 0.001, 0.009, 0.003, 0.008, 0.0011, 0.0007), ('OAO + onset vs. OAO', 0.0104, -0.0, 0.021, 0.006, 0.015, -0.0051, 0.0094), ('MDBT + onset vs. MDBT', 0.0068, 0.003, 0.011, 0.003, 0.01, -0.0099, 0.0054), ('DCEG + onset vs. DCEG', 0.0095, 0.005, 0.014, 0.006, 0.013, -0.0057, 0.0067), ('MLTE + onset vs. MLTE', 0.0062, 0.001, 0.011, 0.003, 0.009, -0.0124, 0.0028), ('Raw + MLTE vector vs. MLTE', 0.0044, -0.003, 0.011, -0.0, 0.009, -0.0018, 0.0028)]
MS_CLF31 = [('Raw + onset vs. Raw', 0.0234, 0.016, 0.031, 0.018, 0.028, 0.0245, 0.017, 0.032, 0.019, 0.03), ('OAO vs. Raw', 0.0132, 0.006, 0.021, 0.009, 0.017, 0.0165, 0.006, 0.028, 0.011, 0.022), ('MDBT vs. Raw', 0.0191, 0.012, 0.026, 0.014, 0.024, 0.0179, 0.009, 0.026, 0.012, 0.024), ('DCEG vs. Raw', 0.0163, 0.01, 0.023, 0.012, 0.021, 0.0176, 0.007, 0.029, 0.011, 0.024), ('MLTE vs. Raw', 0.0226, 0.015, 0.031, 0.017, 0.028, 0.0224, 0.011, 0.033, 0.015, 0.029), ('Random graph vs. Raw', -0.0005, -0.001, 0.0, -0.001, 0.0, -0.0055, -0.012, 0.001, -0.008, -0.003), ('Age-and-sex graph vs. Raw', -0.0005, -0.002, 0.001, -0.001, 0.001, -0.0033, -0.01, 0.003, -0.006, -0.0), ('Shuffled history (DCEG) vs. Raw', 0.0002, -0.001, 0.001, -0.001, 0.001, -0.004, -0.01, 0.002, -0.007, -0.001), ('OAO vs. Random graph', 0.0137, 0.006, 0.021, 0.009, 0.018, 0.022, 0.01, 0.034, 0.016, 0.028), ('MDBT vs. Random graph', 0.0196, 0.012, 0.027, 0.014, 0.024, 0.0234, 0.013, 0.033, 0.017, 0.03), ('DCEG vs. Random graph', 0.0168, 0.01, 0.024, 0.012, 0.021, 0.0231, 0.012, 0.034, 0.016, 0.03), ('MLTE vs. Random graph', 0.0231, 0.015, 0.031, 0.017, 0.029, 0.0279, 0.017, 0.039, 0.02, 0.035), ('OAO vs. Age-and-sex graph', 0.0137, 0.005, 0.022, 0.009, 0.018, 0.0198, 0.006, 0.033, 0.014, 0.026), ('MDBT vs. Age-and-sex graph', 0.0196, 0.012, 0.027, 0.014, 0.024, 0.0212, 0.012, 0.03, 0.015, 0.028), ('DCEG vs. Age-and-sex graph', 0.0168, 0.01, 0.024, 0.012, 0.021, 0.0209, 0.006, 0.036, 0.014, 0.027), ('MLTE vs. Age-and-sex graph', 0.0231, 0.015, 0.032, 0.018, 0.028, 0.0257, 0.013, 0.039, 0.019, 0.032), ('OAO vs. Shuffled history (DCEG)', 0.013, 0.005, 0.021, 0.009, 0.017, 0.0206, 0.008, 0.033, 0.015, 0.026), ('MDBT vs. Shuffled history (DCEG)', 0.0189, 0.011, 0.026, 0.014, 0.024, 0.0219, 0.011, 0.033, 0.016, 0.028), ('DCEG vs. Shuffled history (DCEG)', 0.0161, 0.009, 0.023, 0.011, 0.021, 0.0216, 0.009, 0.034, 0.015, 0.028), ('MLTE vs. Shuffled history (DCEG)', 0.0224, 0.014, 0.031, 0.017, 0.028, 0.0265, 0.014, 0.039, 0.019, 0.033), ('Raw + onset vs. OAO', 0.0102, -0.001, 0.021, 0.005, 0.015, 0.008, -0.003, 0.019, 0.003, 0.013), ('Raw + onset vs. MDBT', 0.0043, -0.001, 0.01, -0.0, 0.009, 0.0066, -0.001, 0.014, 0.002, 0.012), ('Raw + onset vs. DCEG', 0.0071, 0.002, 0.012, 0.002, 0.012, 0.0069, -0.002, 0.016, 0.001, 0.013), ('Raw + onset vs. MLTE', 0.0008, -0.006, 0.008, -0.004, 0.005, 0.0021, -0.006, 0.01, -0.004, 0.008), ('OAO + onset vs. Raw + onset', 0.0002, -0.002, 0.002, -0.001, 0.001, 0.0014, -0.008, 0.011, -0.002, 0.005), ('MDBT + onset vs. Raw + onset', 0.0025, -0.002, 0.007, 0.0, 0.005, -0.0012, -0.008, 0.005, -0.005, 0.003), ('DCEG + onset vs. Raw + onset', 0.0024, -0.0, 0.005, 0.0, 0.005, -0.0003, -0.007, 0.006, -0.005, 0.004), ('MLTE + onset vs. Raw + onset', 0.0054, 0.001, 0.009, 0.003, 0.008, 0.0007, -0.005, 0.006, -0.004, 0.006), ('OAO + onset vs. OAO', 0.0104, -0.0, 0.021, 0.006, 0.015, 0.0094, 0.004, 0.015, 0.005, 0.013), ('MDBT + onset vs. MDBT', 0.0068, 0.003, 0.011, 0.003, 0.01, 0.0054, 0.001, 0.01, 0.002, 0.009), ('DCEG + onset vs. DCEG', 0.0095, 0.005, 0.014, 0.006, 0.013, 0.0067, 0.002, 0.012, 0.003, 0.01), ('MLTE + onset vs. MLTE', 0.0062, 0.001, 0.011, 0.003, 0.009, 0.0028, -0.001, 0.006, -0.001, 0.006), ('OAO vs. DCEG', -0.0031, -0.013, 0.007, -0.008, 0.001, -0.001, -0.01, 0.008, -0.006, 0.004), ('OAO vs. MDBT', -0.0059, -0.014, 0.002, -0.011, -0.001, -0.0014, -0.01, 0.007, -0.006, 0.004), ('DCEG vs. MDBT', -0.0028, -0.01, 0.004, -0.007, 0.002, -0.0004, -0.01, 0.01, -0.006, 0.005), ('MLTE vs. OAO', 0.0094, 0.003, 0.016, 0.005, 0.014, 0.0059, 0.001, 0.011, 0.001, 0.011), ('MLTE vs. DCEG', 0.0063, -0.001, 0.013, 0.002, 0.01, 0.0049, -0.003, 0.013, -0.0, 0.01), ('MLTE vs. MDBT', 0.0035, -0.001, 0.008, -0.001, 0.008, 0.0045, -0.005, 0.014, -0.001, 0.01), ('OAO + onset vs. DCEG + onset', -0.0023, -0.006, 0.001, -0.004, -0.0, 0.0017, -0.006, 0.009, -0.003, 0.006), ('OAO + onset vs. MDBT + onset', -0.0023, -0.006, 0.001, -0.004, -0.0, 0.0027, -0.004, 0.01, -0.001, 0.006), ('DCEG + onset vs. MDBT + onset', -0.0, -0.005, 0.005, -0.003, 0.003, 0.001, -0.007, 0.009, -0.003, 0.005), ('Raw + MLTE vector vs. Raw', 0.027, 0.018, 0.036, 0.02, 0.034, 0.0253, 0.016, 0.035, 0.018, 0.032), ('Raw + MLTE vector vs. MLTE', 0.0044, -0.003, 0.011, -0.0, 0.009, 0.0028, -0.004, 0.009, -0.002, 0.007), ('Raw + MLTE vector vs. Raw + onset', 0.0036, -0.003, 0.011, -0.001, 0.008, 0.0008, -0.006, 0.007, -0.005, 0.006)]
MS_CLFC = [('Raw', 0.0004, -0.0, 0.001, -0.0029, -0.007, 0.001, 0.008, 0.004, 0.012), ('Raw + onset', 0.0145, 0.01, 0.019, -0.004, -0.008, 0.001, 0.0034, 0.0, 0.007), ('OAO', 0.0008, -0.001, 0.002, -0.0062, -0.012, -0.001, 0.0094, 0.005, 0.014), ('MDBT', 0.0026, -0.0, 0.005, -0.0017, -0.006, 0.003, 0.0075, 0.004, 0.012), ('DCEG', 0.0041, 0.001, 0.007, -0.0042, -0.009, 0.001, 0.0081, 0.005, 0.012), ('MLTE', 0.0002, -0.002, 0.003, -0.0027, -0.007, 0.002, 0.0083, 0.005, 0.012), ('Random graph', 0.0058, 0.003, 0.008, 0.0021, -0.003, 0.007, 0.0076, 0.003, 0.012), ('Age-and-sex graph', 0.0076, 0.004, 0.011, -0.0001, -0.005, 0.005, 0.0067, 0.002, 0.011), ('Shuffled history (DCEG)', 0.0059, 0.003, 0.008, 0.0014, -0.003, 0.006, 0.0088, 0.004, 0.013), ('OAO + onset', 0.0162, 0.011, 0.021, -0.0053, -0.01, -0.0, 0.0085, 0.005, 0.012), ('MDBT + onset', 0.0193, 0.015, 0.024, -0.0003, -0.005, 0.004, 0.0064, 0.003, 0.01), ('DCEG + onset', 0.0193, 0.014, 0.024, -0.0013, -0.006, 0.004, 0.0053, 0.002, 0.008), ('MLTE + onset', 0.0188, 0.014, 0.024, 0.0007, -0.004, 0.005, 0.0075, 0.004, 0.011), ('Raw + MLTE vector', 0.0064, 0.003, 0.01, -0.0012, -0.005, 0.003, 0.009, 0.005, 0.013)]
MS_RCV_TUNED = [('Raw + onset vs. Raw', 0.0222, 30, 0.0165, 0.028, 0.0155, 0.0313, 0.0191, 30, 0.0117, 0.0265), ('OAO vs. Raw', 0.0133, 29, 0.0079, 0.0188, 0.0055, 0.021, 0.0095, 30, 0.0054, 0.0137), ('MDBT vs. Raw', 0.0194, 30, 0.014, 0.0248, 0.0117, 0.0265, 0.0138, 30, 0.01, 0.0176), ('DCEG vs. Raw', 0.0163, 30, 0.011, 0.0217, 0.0097, 0.023, 0.0134, 30, 0.008, 0.0188), ('MLTE vs. Raw', 0.0233, 30, 0.0176, 0.029, 0.0146, 0.0307, 0.0158, 30, 0.0107, 0.0209), ('Random graph vs. Raw', -0.0005, 7, -0.0013, 0.0003, -0.0014, 0.0005, 0.0, 13, -0.0008, 0.0009), ('Age-and-sex graph vs. Raw', -0.0003, 15, -0.0014, 0.0008, -0.0023, 0.0013, -0.0001, 15, -0.0011, 0.001), ('Shuffled history (DCEG) vs. Raw', -0.0002, 14, -0.0012, 0.0009, -0.001, 0.0015, -0.0003, 13, -0.0011, 0.0005), ('OAO vs. Random graph', 0.0138, 29, 0.0081, 0.0196, 0.006, 0.0214, 0.0095, 30, 0.0054, 0.0136), ('MDBT vs. Random graph', 0.0199, 30, 0.0143, 0.0255, 0.0122, 0.0269, 0.0138, 30, 0.0099, 0.0176), ('DCEG vs. Random graph', 0.0168, 30, 0.0113, 0.0224, 0.0099, 0.0237, 0.0134, 30, 0.0081, 0.0187), ('MLTE vs. Random graph', 0.0238, 30, 0.018, 0.0296, 0.0151, 0.0311, 0.0158, 30, 0.0109, 0.0206), ('OAO vs. Age-and-sex graph', 0.0136, 29, 0.0079, 0.0194, 0.0052, 0.0223, 0.0096, 30, 0.0051, 0.0141), ('MDBT vs. Age-and-sex graph', 0.0197, 30, 0.0141, 0.0253, 0.0121, 0.027, 0.0139, 30, 0.0097, 0.018), ('DCEG vs. Age-and-sex graph', 0.0166, 30, 0.0109, 0.0224, 0.0095, 0.024, 0.0135, 30, 0.0079, 0.019), ('MLTE vs. Age-and-sex graph', 0.0236, 30, 0.0177, 0.0295, 0.0146, 0.0316, 0.0159, 30, 0.0106, 0.0211), ('OAO vs. Shuffled history (DCEG)', 0.0135, 29, 0.0079, 0.0191, 0.0053, 0.0207, 0.0098, 30, 0.0058, 0.0139), ('MDBT vs. Shuffled history (DCEG)', 0.0196, 30, 0.014, 0.0251, 0.0113, 0.0264, 0.0141, 30, 0.0101, 0.0181), ('DCEG vs. Shuffled history (DCEG)', 0.0165, 30, 0.0111, 0.0219, 0.0092, 0.023, 0.0137, 30, 0.0083, 0.0191), ('MLTE vs. Shuffled history (DCEG)', 0.0235, 30, 0.0177, 0.0292, 0.0142, 0.0305, 0.0161, 30, 0.011, 0.0212), ('Raw + onset vs. OAO', 0.0089, 26, 0.0028, 0.0149, -0.0006, 0.021, 0.0096, 26, 0.0033, 0.0158), ('Raw + onset vs. MDBT', 0.0028, 19, -0.0019, 0.0075, -0.0013, 0.01, 0.0053, 22, -0.0015, 0.0121), ('Raw + onset vs. DCEG', 0.0059, 25, 0.0017, 0.0101, 0.0019, 0.0123, 0.0057, 25, 0.0008, 0.0106), ('OAO + onset vs. Raw + onset', 0.0003, 17, -0.0017, 0.0023, -0.0016, 0.002, 0.0002, 20, -0.001, 0.0014), ('MDBT + onset vs. Raw + onset', 0.0025, 24, -0.0003, 0.0053, -0.0018, 0.0067, 0.0007, 21, -0.0018, 0.0032), ('DCEG + onset vs. Raw + onset', 0.0025, 24, -0.0002, 0.0053, -0.0002, 0.0051, 0.003, 26, 0.0005, 0.0056), ('MLTE + onset vs. Raw + onset', 0.0065, 29, 0.003, 0.0101, 0.0014, 0.0094, 0.0036, 26, 0.0013, 0.0059), ('OAO + onset vs. OAO', 0.0092, 28, 0.0033, 0.015, -0.0, 0.0207, 0.0098, 28, 0.0036, 0.016), ('MDBT + onset vs. MDBT', 0.0053, 26, 0.0016, 0.0091, 0.003, 0.0105, 0.006, 24, 0.0001, 0.0119), ('DCEG + onset vs. DCEG', 0.0084, 30, 0.0046, 0.0123, 0.0051, 0.0139, 0.0087, 29, 0.0033, 0.0141), ('MLTE + onset vs. MLTE', 0.0055, 27, 0.0015, 0.0094, 0.0013, 0.0112, 0.0069, 27, 0.0009, 0.0128), ('OAO vs. DCEG', -0.003, 11, -0.0089, 0.003, -0.0132, 0.007, -0.0039, 8, -0.0086, 0.0009), ('OAO vs. MDBT', -0.0061, 4, -0.0109, -0.0012, -0.0137, 0.002, -0.0043, 5, -0.0092, 0.0007), ('DCEG vs. MDBT', -0.0031, 14, -0.0084, 0.0022, -0.0099, 0.0043, -0.0004, 14, -0.0052, 0.0044), ('MLTE vs. OAO', 0.01, 29, 0.0051, 0.0148, 0.0026, 0.0161, 0.0062, 29, 0.002, 0.0105), ('MLTE vs. DCEG', 0.007, 25, 0.0022, 0.0117, -0.0006, 0.0132, 0.0024, 21, -0.0018, 0.0065), ('MLTE vs. MDBT', 0.0039, 24, -0.0, 0.0078, -0.0012, 0.0082, 0.002, 20, -0.0027, 0.0067), ('OAO + onset vs. DCEG + onset', -0.0023, 7, -0.0048, 0.0002, -0.0056, 0.0011, -0.0028, 4, -0.0054, -0.0003), ('OAO + onset vs. MDBT + onset', -0.0022, 6, -0.0049, 0.0005, -0.0057, 0.0012, -0.0005, 10, -0.0031, 0.0021), ('DCEG + onset vs. MDBT + onset', 0.0, 15, -0.0035, 0.0035, -0.0055, 0.0054, 0.0024, 20, -0.0014, 0.0062), ('MLTE + onset vs. OAO + onset', 0.0062, 29, 0.0031, 0.0094, 0.0024, 0.0081, 0.0034, 26, 0.0012, 0.0056), ('MLTE + onset vs. DCEG + onset', 0.004, 25, 0.0006, 0.0074, -0.0016, 0.0075, 0.0005, 17, -0.0022, 0.0032), ('MLTE + onset vs. MDBT + onset', 0.004, 24, 0.0012, 0.0069, -0.0003, 0.0062, 0.0029, 24, -0.0005, 0.0064), ('Raw + MLTE vector vs. Raw', 0.0266, 30, 0.0193, 0.0339, 0.0177, 0.0363, 0.0198, 30, 0.0136, 0.0261), ('Raw + MLTE vector vs. MLTE', 0.0033, 21, -0.0013, 0.008, -0.0026, 0.0114, 0.004, 24, 0.0005, 0.0076)]
MS_RCV_DEFAULT = [('Raw + onset vs. Raw', 0.0084, 23, -0.0006, 0.0175, -0.0003, 0.019, 0.0106, 27, 0.0025, 0.0186), ('OAO vs. Raw', 0.0129, 28, 0.0062, 0.0196, 0.0049, 0.0209, 0.0097, 30, 0.0046, 0.0148), ('MDBT vs. Raw', 0.0181, 30, 0.0119, 0.0243, 0.0081, 0.0258, 0.0139, 30, 0.0091, 0.0187), ('DCEG vs. Raw', 0.0125, 29, 0.006, 0.0191, 0.0054, 0.02, 0.0124, 29, 0.0064, 0.0184), ('MLTE vs. Raw', 0.0229, 30, 0.0152, 0.0305, 0.0131, 0.0325, 0.0176, 30, 0.0115, 0.0237), ('Random graph vs. Raw', -0.0067, 3, -0.0106, -0.0028, -0.0118, 0.0001, -0.0022, 3, -0.0037, -0.0006), ('Age-and-sex graph vs. Raw', -0.0072, 3, -0.0121, -0.0022, -0.016, 0.0006, -0.0035, 7, -0.0066, -0.0004), ('Shuffled history (DCEG) vs. Raw', -0.0046, 4, -0.0078, -0.0014, -0.0086, -0.0019, -0.0018, 5, -0.0038, 0.0001), ('OAO vs. Random graph', 0.0196, 29, 0.012, 0.0272, 0.0101, 0.0274, 0.0119, 30, 0.0065, 0.0172), ('MDBT vs. Random graph', 0.0248, 30, 0.0175, 0.0321, 0.0142, 0.0314, 0.0161, 30, 0.011, 0.0212), ('DCEG vs. Random graph', 0.0192, 30, 0.0123, 0.0262, 0.0116, 0.0255, 0.0145, 30, 0.0085, 0.0206), ('MLTE vs. Random graph', 0.0296, 30, 0.0212, 0.0379, 0.0213, 0.0361, 0.0197, 30, 0.0135, 0.026), ('OAO vs. Age-and-sex graph', 0.0201, 30, 0.0118, 0.0284, 0.008, 0.0332, 0.0132, 30, 0.0079, 0.0185), ('MDBT vs. Age-and-sex graph', 0.0253, 30, 0.0177, 0.0329, 0.0135, 0.0358, 0.0174, 30, 0.0123, 0.0226), ('DCEG vs. Age-and-sex graph', 0.0197, 29, 0.011, 0.0285, 0.0079, 0.0329, 0.0159, 30, 0.0108, 0.0209), ('MLTE vs. Age-and-sex graph', 0.0301, 30, 0.0209, 0.0392, 0.0174, 0.0437, 0.0211, 30, 0.015, 0.0272), ('OAO vs. Shuffled history (DCEG)', 0.0175, 30, 0.0109, 0.0242, 0.009, 0.0273, 0.0115, 30, 0.0062, 0.0169), ('MDBT vs. Shuffled history (DCEG)', 0.0227, 30, 0.0168, 0.0286, 0.0133, 0.0311, 0.0158, 30, 0.0106, 0.0209), ('DCEG vs. Shuffled history (DCEG)', 0.0172, 30, 0.0106, 0.0237, 0.0116, 0.0243, 0.0142, 30, 0.0077, 0.0207), ('MLTE vs. Shuffled history (DCEG)', 0.0275, 30, 0.0197, 0.0353, 0.0179, 0.0382, 0.0194, 30, 0.013, 0.0258), ('Raw + onset vs. OAO', -0.0045, 10, -0.0128, 0.0038, -0.016, 0.0089, 0.0008, 16, -0.0074, 0.0091), ('Raw + onset vs. MDBT', -0.0097, 4, -0.0165, -0.0029, -0.0157, 0.0005, -0.0034, 9, -0.0106, 0.0038), ('Raw + onset vs. DCEG', -0.0041, 8, -0.0108, 0.0025, -0.0125, 0.0058, -0.0018, 12, -0.009, 0.0054), ('OAO + onset vs. Raw + onset', -0.0019, 11, -0.0058, 0.0019, -0.0069, 0.0038, -0.001, 10, -0.0027, 0.0007), ('MDBT + onset vs. Raw + onset', -0.0011, 11, -0.0046, 0.0024, -0.0075, 0.0028, -0.0002, 12, -0.0022, 0.0018), ('DCEG + onset vs. Raw + onset', -0.003, 7, -0.0064, 0.0005, -0.0065, 0.0018, -0.001, 12, -0.0031, 0.0011), ('MLTE + onset vs. Raw + onset', 0.0008, 17, -0.0026, 0.0042, -0.0041, 0.0063, 0.0003, 17, -0.0016, 0.0022), ('OAO + onset vs. OAO', -0.0064, 7, -0.0133, 0.0005, -0.0151, 0.0048, -0.0001, 11, -0.0076, 0.0074), ('MDBT + onset vs. MDBT', -0.0108, 2, -0.0162, -0.0054, -0.0152, -0.0046, -0.0036, 5, -0.0109, 0.0037), ('DCEG + onset vs. DCEG', -0.0071, 5, -0.0132, -0.001, -0.0125, 0.0011, -0.0028, 8, -0.0091, 0.0034), ('MLTE + onset vs. MLTE', -0.0137, 3, -0.0205, -0.0068, -0.0208, -0.004, -0.0067, 7, -0.014, 0.0006), ('OAO vs. DCEG', 0.0004, 17, -0.0069, 0.0076, -0.0106, 0.011, -0.0027, 9, -0.0082, 0.0029), ('OAO vs. MDBT', -0.0052, 6, -0.0116, 0.0012, -0.0124, 0.0043, -0.0042, 8, -0.0099, 0.0015), ('DCEG vs. MDBT', -0.0056, 10, -0.0122, 0.0011, -0.014, 0.0055, -0.0016, 13, -0.0065, 0.0034), ('MLTE vs. OAO', 0.01, 27, 0.0033, 0.0166, 0.0029, 0.017, 0.0079, 29, 0.0032, 0.0126), ('MLTE vs. DCEG', 0.0103, 29, 0.0042, 0.0165, 0.0016, 0.0187, 0.0052, 27, -0.0004, 0.0108), ('MLTE vs. MDBT', 0.0048, 22, -0.0002, 0.0097, -0.0016, 0.0134, 0.0036, 21, -0.0024, 0.0096), ('OAO + onset vs. DCEG + onset', 0.001, 16, -0.0039, 0.0059, -0.0063, 0.0078, 0.0, 15, -0.0024, 0.0024), ('OAO + onset vs. MDBT + onset', -0.0008, 13, -0.005, 0.0034, -0.0029, 0.0043, -0.0008, 14, -0.0037, 0.0022), ('DCEG + onset vs. MDBT + onset', -0.0019, 13, -0.0075, 0.0038, -0.0075, 0.0074, -0.0008, 14, -0.004, 0.0024), ('MLTE + onset vs. OAO + onset', 0.0027, 20, -0.0021, 0.0075, -0.0037, 0.0091, 0.0013, 20, -0.0011, 0.0036), ('MLTE + onset vs. DCEG + onset', 0.0037, 21, -0.0013, 0.0088, -0.004, 0.0109, 0.0013, 20, -0.0012, 0.0038), ('MLTE + onset vs. MDBT + onset', 0.0019, 19, -0.0025, 0.0063, -0.0035, 0.0104, 0.0005, 20, -0.002, 0.003), ('Raw + MLTE vector vs. Raw', 0.0201, 30, 0.0122, 0.0279, 0.0081, 0.0339, 0.0169, 30, 0.009, 0.0249), ('Raw + MLTE vector vs. MLTE', -0.0028, 10, -0.0093, 0.0037, -0.0099, 0.0063, -0.0007, 16, -0.0067, 0.0054)]
MS_ARMS26 = [('Raw + presence-only', 265, 0.6789, 0.0144, 0.6693, 0.688, 0.115, 0.0123, 0.6699, 0.678), ('Raw + code count', 63, 0.6757, 0.0141, 0.6664, 0.6848, 0.107, 0.0099, 0.6747, 0.6742), ('Raw + counts by content group', 68, 0.6819, 0.0148, 0.6723, 0.6909, 0.1139, 0.0084, 0.6815, 0.6788), ('Raw + onset, -1 coding', 265, 0.67, 0.0157, 0.6602, 0.6794, 0.1077, 0.0079, 0.6677, 0.6769), ('Raw + onset, no F41/F43', 464, 0.6729, 0.0146, 0.6636, 0.6821, 0.1082, 0.0115, 0.6578, 0.6689), ('Raw + onset, no primary-care-type', 410, 0.6787, 0.015, 0.6691, 0.6878, 0.1143, 0.0124, 0.6662, 0.6749), ('MDBT, no F41/F43', 186, 0.6703, 0.0129, 0.661, 0.6794, 0.1063, 0.0101, 0.6684, 0.6626), ('DCEG, no F41/F43', 186, 0.6706, 0.0145, 0.6611, 0.6799, 0.1095, 0.0116, 0.6654, 0.6682), ('OAO, no F41/F43', 186, 0.6657, 0.0145, 0.6566, 0.6752, 0.1043, 0.0135, 0.6651, 0.6678), ('OAO + onset, both no F41/F43', 588, 0.6728, 0.0139, 0.6635, 0.6822, 0.1078, 0.0118, 0.6549, 0.6721), ('MDBT + onset, both no F41/F43', 588, 0.6749, 0.0124, 0.6655, 0.6839, 0.1103, 0.0114, 0.6552, 0.6674), ('DCEG + onset, both no F41/F43', 588, 0.675, 0.0129, 0.6657, 0.6842, 0.1117, 0.0118, 0.655, 0.6717), ('OAO, 51-candidate pool', 186, 0.6639, 0.0182, 0.6544, 0.6733, 0.1023, 0.0117, 0.6629, 0.6679), ('Weighted Jaccard, pool 500 (untimed)', 186, 0.6679, 0.0177, 0.6584, 0.6772, 0.1063, 0.0122, 0.6663, 0.6741), ('Weighted Jaccard (untimed) + presence', 389, 0.6784, 0.0146, 0.6688, 0.6877, 0.1156, 0.0136, 0.6654, 0.6777), ('Raw + MDBT descriptor', 188, 0.6827, 0.0118, 0.6735, 0.6921, 0.115, 0.0077, 0.6797, 0.6811), ('Raw + DCEG embedding', 126, 0.669, 0.0129, 0.6598, 0.6785, 0.1022, 0.0121, 0.667, 0.6725), ('MDBT descriptor only', 126, 0.6281, 0.0103, 0.6181, 0.6378, 0.0903, 0.0045, 0.6239, 0.6259), ('DCEG embedding only', 64, 0.5939, 0.0105, 0.5844, 0.6039, 0.0628, 0.005, 0.5916, 0.6107), ('MDBT + MDBT descriptor', 312, 0.6828, 0.0119, 0.6735, 0.6923, 0.1152, 0.0078, 0.6756, 0.677), ('DCEG + DCEG embedding', 250, 0.6736, 0.0124, 0.6644, 0.6829, 0.1103, 0.0132, 0.6696, 0.666), ('Random graph + MDBT descriptor', 312, 0.6827, 0.0118, 0.6735, 0.6921, 0.115, 0.0077, 0.6746, 0.6778), ('Random graph + DCEG embedding', 250, 0.6688, 0.013, 0.6596, 0.6783, 0.1022, 0.0121, 0.6616, 0.6621), ('Random graph + onset', 592, 0.6782, 0.0142, 0.6686, 0.6874, 0.114, 0.0117, 0.6612, 0.672), ('Feature-kNN control', 186, 0.6537, 0.0164, 0.6446, 0.6631, 0.0942, 0.0099, 0.6492, 0.6397), ('Node-shuffled DCEG graph', 186, 0.6547, 0.016, 0.6455, 0.6639, 0.0953, 0.0106, 0.6474, 0.6457)]
MS_NOF = [('OAO, no F41/F43 vs. OAO', -0.0026, -0.004, -0.0013, 0.001, -0.0025, -0.004, -0.0008, 0.019), ('MDBT, no F41/F43 vs. MDBT', -0.004, -0.0064, -0.0014, 0.016, -0.0033, -0.0062, -0.0003, 0.185), ('DCEG, no F41/F43 vs. DCEG', -0.0009, -0.0019, 0.0002, 0.258, -0.002, -0.0041, 0.0001, 0.241), ('OAO, no F41/F43 vs. Raw', 0.0106, 0.0064, 0.0146, -0.001, 0.0104, 0.0053, 0.0153, -0.001), ('OAO, no F41/F43 vs. Random graph', 0.0111, 0.0067, 0.0151, -0.001, 0.0163, 0.0106, 0.0217, -0.001), ('MDBT, no F41/F43 vs. Raw', 0.0152, 0.0107, 0.0195, -0.001, 0.0136, 0.0079, 0.0192, -0.001), ('MDBT, no F41/F43 vs. Random graph', 0.0156, 0.0111, 0.02, -0.001, 0.0195, 0.0134, 0.0254, -0.001), ('DCEG, no F41/F43 vs. Raw', 0.0155, 0.0107, 0.0199, -0.001, 0.0107, 0.0051, 0.0162, 0.002), ('DCEG, no F41/F43 vs. Random graph', 0.0159, 0.0112, 0.0203, -0.001, 0.0166, 0.0107, 0.0223, -0.001), ('OAO + onset, both no F41/F43 vs. OAO + onset', -0.0059, -0.0086, -0.0031, -0.001, -0.0076, -0.0102, -0.0049, -0.001), ('MDBT + onset, both no F41/F43 vs. MDBT + onset', -0.0061, -0.0086, -0.0035, -0.001, -0.0065, -0.0097, -0.0034, -0.001), ('DCEG + onset, both no F41/F43 vs. DCEG + onset', -0.006, -0.0088, -0.0033, -0.001, -0.0067, -0.0096, -0.0038, -0.001), ('OAO + onset, both no F41/F43 vs. Raw + onset, no F41/F43', -0.0001, -0.0013, 0.0012, 0.935, -0.0029, -0.0057, -0.0002, 0.185), ('MDBT + onset, both no F41/F43 vs. Raw + onset, no F41/F43', 0.002, -0.0002, 0.0042, 0.258, -0.0026, -0.0061, 0.0006, 0.241), ('DCEG + onset, both no F41/F43 vs. Raw + onset, no F41/F43', 0.0021, -0.0001, 0.0044, 0.258, -0.0028, -0.0059, 0.0001, 0.241), ('OAO + onset, both no F41/F43 vs. OAO, no F41/F43', 0.0071, 0.0036, 0.0107, -0.001, -0.0103, -0.0158, -0.0045, 0.004), ('MDBT + onset, both no F41/F43 vs. MDBT, no F41/F43', 0.0046, 0.0014, 0.0077, 0.019, -0.0132, -0.0188, -0.0078, -0.001), ('DCEG + onset, both no F41/F43 vs. DCEG, no F41/F43', 0.0044, 0.0016, 0.0072, 0.016, -0.0104, -0.016, -0.0051, 0.002), ('Raw + onset, no F41/F43 vs. Raw', 0.0178, 0.0132, 0.0221, -0.001, 0.0031, -0.0036, 0.0094, 0.347)]

# Part 3 — Paired contrasts on the main run
For every contrast (first minus second representation) and each classifier: mean paired fold difference, **corrected resampled-t** 95% interval (variance factor 1/J + n_test/n_train, df = J−1) and p-value, Holm adjustment within the 31 main comparisons and within the MLTE family, and the stored **patient-level bootstrap** interval (looked up in the bootstrap CSVs; flipped when the file stores the reverse order).

In [ ]:
SUFFIX = {TLR: '', DLR: '_logisticregression', DCB: '_catboost', TCB: '_catboosttuned'}
def _bootfile(stem, model, met):
    return read(f'bootstrap_{stem}_{met}{SUFFIX[model]}.csv', index_col=0)
BOOT = {}
for stem in ('main', 'mlte', 'nof', 'sens'):
    for model in CLF.values():
        for met in ('auroc', 'aupr'):
            try: BOOT[(stem, model, met)] = _bootfile(stem, model, met)
            except Exception as e: pass
def _split(c): 
    a, b = c.split(' vs ')
    return a.strip(), b.strip()
def boot_lookup(stem, model, met, a, b):
    """-> (delta, lo, hi, p_z, p_holm) or None; flips sign if the file stores b vs a"""
    df = BOOT.get((stem, model, met))
    if df is None: return None
    for c in df.index:
        x, y = _split(c)
        if (x, y) == (a, b): r = df.loc[c]; return r.delta, r.lo, r.hi, r.p_z, r.p_holm
        if (x, y) == (b, a): r = df.loc[c]; return -r.delta, -r.hi, -r.lo, r.p_z, r.p_holm
    return None

MAIN_PAIRS = [_split(c) for c in BOOT[('main', TLR, 'auroc')].index]
MLTE_PAIRS = [_split(c) for c in BOOT[('mlte', TLR, 'auroc')].index]                      # 13 MLTE comparisons with bootstrap intervals (tab:clf31)
MLTE_RCV_PAIRS = [_split(c) for c in read('rcv_mlte_auroc.csv', index_col=0).index]         # 14 MLTE comparisons of the repeated-CV run (tab:paired-*)
_have = set(MAIN_PAIRS) | set(MLTE_PAIRS) | {(b, a) for a, b in MAIN_PAIRS + MLTE_PAIRS}
MLTE_EXTRA = [p for p in MLTE_RCV_PAIRS if p not in _have]                                  # repeated-CV only
print(len(MAIN_PAIRS), 'main;', len(MLTE_PAIRS), 'MLTE with bootstrap;', len(MLTE_RCV_PAIRS), 'MLTE in repeated CV;', 'repeated-CV-only:', [pair_label(*p) for p in MLTE_EXTRA])

def pair_row(model, a, b, met='auc', stem=None, Wd=None):
    Wd = W if Wd is None else Wd
    d = (Wd[(model, met)][a] - Wd[(model, met)][b]).values
    m, lo, hi, p = corrected_t(d)
    r = dict(delta=m, ct_lo=lo, ct_hi=hi, ct_p=p, folds_ahead=int((d > 0).sum()), n_folds=len(d))
    bl = None
    for st in (('main', 'mlte') if stem else ()):
        bl = boot_lookup(st, model, {'auc': 'auroc', 'aupr': 'aupr'}[met], a, b)
        if bl: break
    r.update(dict(zip(('boot_delta', 'boot_lo', 'boot_hi', 'boot_p', 'boot_holm'), bl if bl else (np.nan,) * 5)))
    return r

CONTR = {}
for model in CLF.values():
    rows = {}
    for fam, pairs in (('main', MAIN_PAIRS), ('mlte', MLTE_PAIRS), ('mlte_rcv_only', MLTE_EXTRA)):
        if not pairs: continue
        fr = pd.DataFrame({(a, b): pair_row(model, a, b, stem=fam) for a, b in pairs}).T
        fr['family'] = fam
        fr['ct_holm'] = holm(fr.ct_p.values)
        fr['ct_bh'] = bh(fr.ct_p.values)
        rows[fam] = fr
    CONTR[model] = pd.concat(rows.values())
    CONTR[model].index = pd.MultiIndex.from_tuples(CONTR[model].index, names=['a', 'b'])
def C(model, a, b):
    """lookup in either order of a stored pair; computes it on the fly if it is not one of the 45"""
    t = CONTR[model]
    if (a, b) in t.index: return t.loc[(a, b)]
    if (b, a) in t.index:
        r = t.loc[(b, a)].copy(); r.delta, r.ct_lo, r.ct_hi = -r.delta, -r.ct_hi, -r.ct_lo; r.boot_delta, r.boot_lo, r.boot_hi = -r.boot_delta, -r.boot_hi, -r.boot_lo
        r.folds_ahead = r.n_folds - r.folds_ahead; return r
    return pd.Series(pair_row(model, a, b))

def contrast_table(label_pairs, models, cols=('delta', 'ct', 'boot')):
    out = {}
    for lbl in label_pairs:
        a, b = pair(lbl); rec = {}
        for m in models:
            r = C(m, a, b)
            rec[(CLF_NAME[m], 'Δ')] = r.delta
            if 'ct' in cols: rec[(CLF_NAME[m], 'corrected-t')] = ci(r.ct_lo, r.ct_hi)
            if 'boot' in cols: rec[(CLF_NAME[m], 'bootstrap')] = ci(r.boot_lo, r.boot_hi) if not np.isnan(r.boot_lo) else ''
            if 'p' in cols: rec[(CLF_NAME[m], 'Holm p (ct)')] = r.get('ct_holm', np.nan)
        out[lbl] = rec
    return pd.DataFrame(out).T

# ---------- tab:keycontrasts
lab_key = [r[0] for r in MS_KEY]
kc = contrast_table(lab_key, [TLR]).join(contrast_table(lab_key, [DLR, TCB], cols=('delta',)))
show('keycontrasts', kc, 'tab:keycontrasts — principal paired contrasts in AUROC (first minus second)')
for (lbl, d, cl, ch, bl, bh_, dd, dt) in MS_KEY:
    a, b = pair(lbl); r = C(TLR, a, b)
    claim('tab:keycontrasts', f'{lbl}: Δ tuned LR', d, r.delta, 4, True)
    claim('tab:keycontrasts', f'{lbl}: corrected-t lower', cl, r.ct_lo, 3, True); claim('tab:keycontrasts', f'{lbl}: corrected-t upper', ch, r.ct_hi, 3, True)
    claim('tab:keycontrasts', f'{lbl}: bootstrap lower', bl, r.boot_lo, 3, True); claim('tab:keycontrasts', f'{lbl}: bootstrap upper', bh_, r.boot_hi, 3, True)
    claim('tab:keycontrasts', f'{lbl}: Δ default L2', dd, C(DLR, a, b).delta, 4, True); claim('tab:keycontrasts', f'{lbl}: Δ tuned CatBoost', dt, C(TCB, a, b).delta, 4, True)

# ---------- tab:clf31 (tuned LR and tuned CatBoost; 44 comparisons)
lab31 = [r[0] for r in MS_CLF31]
c31 = contrast_table(lab31, [TLR, TCB])
show('clf31', c31, 'tab:clf31 — all main and MLTE comparisons under tuned LR and tuned CatBoost')
for (lbl, d1, l1, h1, bl1, bh1, d2, l2, h2, bl2, bh2) in MS_CLF31:
    a, b = pair(lbl); r1, r2 = C(TLR, a, b), C(TCB, a, b)
    for tag, v, rr in (('Δ tuned LR', d1, r1.delta), ('ct lo tuned LR', l1, r1.ct_lo), ('ct hi tuned LR', h1, r1.ct_hi), ('boot lo tuned LR', bl1, r1.boot_lo), ('boot hi tuned LR', bh1, r1.boot_hi),
                       ('Δ tuned CB', d2, r2.delta), ('ct lo tuned CB', l2, r2.ct_lo), ('ct hi tuned CB', h2, r2.ct_hi), ('boot lo tuned CB', bl2, r2.boot_lo), ('boot hi tuned CB', bh2, r2.boot_hi)):
        claim('tab:clf31', f'{lbl}: {tag}', v, rr, 4 if tag.startswith('Δ') else 3, True)

# ---------- Holm-adjusted results of the main family (tuned LR)
hm = CONTR[TLR].xs('main', level=None, drop_level=False) if False else CONTR[TLR][CONTR[TLR].family == 'main']
hm_show = hm[['delta', 'ct_lo', 'ct_hi', 'ct_p', 'ct_holm', 'ct_bh', 'boot_lo', 'boot_hi', 'boot_p', 'boot_holm']].copy()
hm_show.index = [pair_label(a, b) for a, b in hm_show.index]
show('holm_main', hm_show, 'Tuned LR, 31 main comparisons: corrected-t p-values with Holm / BH, bootstrap z-test p with the stored Holm')
sig = CONTR[TLR][CONTR[TLR].family == 'main']
print('Holm-significant (p<.05) among the 31 main comparisons, tuned LR:', int((sig.ct_holm < .05).sum()))
onset_vs_graph = [C(TLR, 'raw_onset', g) for g in HAND3]
claim_true('Results', 'onset columns vs the three handcrafted graphs: none survives Holm adjustment (corrected-t)',
           all(C(TLR, 'raw_onset', g).ct_holm >= .05 for g in HAND3), 'Holm p = ' + ', '.join(f'{C(TLR, "raw_onset", g).ct_holm:.3f}' for g in HAND3))
claim('Results', 'onset columns - raw features', .023, C(TLR, 'raw_onset', 'raw').delta, 3)
claim('Results', '  interval lower', .016, C(TLR, 'raw_onset', 'raw').ct_lo, 3, True); claim('Results', '  interval upper', .031, C(TLR, 'raw_onset', 'raw').ct_hi, 3, True)
claim_range('Results', 'onset ahead of handcrafted graphs by 0.004-0.010', .004, .010, [C(TLR, 'raw_onset', g).delta for g in HAND3], 3)

# ---------- tab:answers (ranges)
gr_raw = lambda m: [C(m, g, 'raw').delta for g in GRAPHS4]
claim_range('tab:answers Q1', 'graph - raw, tuned LR', .013, .023, gr_raw(TLR), 3, True)
claim_true('tab:answers Q1', 'lower corrected-t limits of the four graphs vs raw exceed +0.005', min(C(TLR, g, 'raw').ct_lo for g in GRAPHS4) > .005,
           f'min lower = {min(C(TLR, g, "raw").ct_lo for g in GRAPHS4):.4f}')
claim_range('tab:answers Q1', 'controls - raw, tuned LR', -.0005, .0002, [C(TLR, c, 'raw').delta for c in CTRL3], 4, True)
claim_range('tab:answers Q1', 'graph - raw, tuned CatBoost', .017, .022, gr_raw(TCB), 3, True)
claim_range('tab:answers Q1', 'graph - raw, default L2', .013, .023, gr_raw(DLR), 3, True)
gc = [(g, c) for g in HAND3 for c in CTRL3] + [('mlte_50', c) for c in CTRL3]
og = {'OAO': 'oaopool500_50', 'MDBT': 'mdbt_50', 'DCEG': 'dceg_50', 'MLTE': 'mlte_50'}
for lab, v in zip(og, (.010, .004, .007, .001)):
    claim('tab:answers Q2', f'onset - {lab}, tuned LR', v, C(TLR, 'raw_onset', og[lab]).delta, 3, True)
claim_range('tab:answers Q2', 'onset - graph, tuned CatBoost', .002, .008, [C(TCB, 'raw_onset', g).delta for g in og.values()], 3, True)
claim_range('tab:answers Q2', 'onset - graph, default L2 (reversed)', -.014, -.003, [C(DLR, 'raw_onset', g).delta for g in og.values()], 3, True)
og2 = dict(zip(og, ('oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset', 'mlte_50_onset')))
for lab, v in zip(og, (.0002, .0025, .0024, .0054)):
    claim('tab:answers Q2', f'{lab} + onset - onset, tuned LR', v, C(TLR, og2[lab], 'raw_onset').delta, 4, True)
claim_range('tab:answers Q2', 'upper limits of graph+onset - onset (corrected-t)', .002, .009, [C(TLR, og2[l], 'raw_onset').ct_hi for l in og], 3, True)
claim_range('tab:answers Q2', 'graph+onset - onset, tuned CatBoost', -.001, .001, [C(TCB, og2[l], 'raw_onset').delta for l in og], 3, True)
claim_range('tab:answers Q2', 'graph+onset - onset, default L2', -.002, .001, [C(DLR, og2[l], 'raw_onset').delta for l in og], 3, True)

# ---------- Which main contrasts are Holm-significant under which classifier? (counts used in the text)
cnt = pd.DataFrame({CLF_NAME[m]: {'corrected-t, Holm<.05': int((CONTR[m][CONTR[m].family == 'main'].ct_holm < .05).sum()),
                                  'bootstrap, Holm<.05': int((CONTR[m][CONTR[m].family == 'main'].boot_holm < .05).sum()),
                                  'sign same as tuned LR (31)': int((np.sign(CONTR[m][CONTR[m].family == 'main'].delta) == np.sign(CONTR[TLR][CONTR[TLR].family == 'main'].delta)).sum())} for m in CLF.values()})
show('holm_counts', cnt, 'Number of the 31 main comparisons that reach Holm p<0.05, by classifier and interval type')

# Part 4 — Classifier contrasts (tab:clfcontrasts)
Tuned LR − default L2, tuned LR − tuned CatBoost and tuned CatBoost − default CatBoost for the 14 main representations; patient-level bootstrap intervals on the same resamples (precomputed files).

In [ ]:
CC = {'tuned_vs_lr': 'Tuned LR − default L2', 'tuned_vs_catboosttuned': 'Tuned LR − tuned CB', 'catboosttuned_vs_catboost': 'Tuned CB − default CB'}
CCD = {k: {met: read(f'bootstrap_{k}_{met}.csv', index_col=0) for met in ('auroc', 'aupr')} for k in CC}
rows = {}
for a in MAIN14:
    rec = {}
    for k, nm in CC.items():
        r = CCD[k]['auroc'].loc[a]
        rec[(nm, 'Δ')] = r.delta; rec[(nm, 'interval')] = ci(r.lo, r.hi); rec[(nm, 'Holm p')] = r.p_holm
    rows[ARM2LABEL[a]] = rec
clfc = pd.DataFrame(rows).T
show('clfcontrasts', clfc, 'tab:clfcontrasts — classifier contrasts in AUROC (bootstrap)')
for (lbl, d1, l1, h1, d2, l2, h2, d3, l3, h3) in MS_CLFC:
    a = LABEL[lbl]
    for (k, d, l, h) in (('tuned_vs_lr', d1, l1, h1), ('tuned_vs_catboosttuned', d2, l2, h2), ('catboosttuned_vs_catboost', d3, l3, h3)):
        r = CCD[k]['auroc'].loc[a]
        claim('tab:clfcontrasts', f'{lbl}: {CC[k]} Δ', d, r.delta, 4, True)
        claim('tab:clfcontrasts', f'{lbl}: {CC[k]} lower', l, r.lo, 3, True); claim('tab:clfcontrasts', f'{lbl}: {CC[k]} upper', h, r.hi, 3, True)
    # bold = Holm p<.05 within the family (11 handcrafted... ) is reported in the table; the Holm p's are shown above
# Q3 statements in tab:answers
d_ = CCD['tuned_vs_lr']['auroc']
claim_range('tab:answers Q3', 'tuned - default L2, onset-containing representations', .015, .019,
            d_.loc[['raw_onset', 'oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset', 'mlte_50_onset'], 'delta'], 3, True,
            note='raw + MLTE vector (0.006) is listed separately in tab:clfcontrasts')
claim_range('tab:answers Q3', 'tuned - default L2, control graphs', .006, .008, d_.loc[CTRL3, 'delta'], 3, True)
claim_true('tab:answers Q3', 'tuned - default L2 for raw, OAO, MDBT, MLTE is at most +0.003', d_.loc[['raw', 'oaopool500_50', 'mdbt_50', 'mlte_50'], 'delta'].max() <= .0035,
           f'max = {d_.loc[["raw", "oaopool500_50", "mdbt_50", "mlte_50"], "delta"].max():.4f}')
claim('tab:answers Q3', 'tuned LR - tuned CB: smallest Holm p over the 14 arms (manuscript: >= 0.24)', .24, CCD['tuned_vs_catboosttuned']['auroc'].loc[MAIN14, 'p_holm'].min(), 2)
aupr_cc = pd.DataFrame({CC[k]: CCD[k]['aupr'].loc[MAIN14, 'delta'] for k in CC}); aupr_cc.index = [ARM2LABEL[a] for a in MAIN14]
show('clfcontrasts_aupr', aupr_cc, 'Same classifier contrasts in AUPR (bootstrap Δ)')

# Part 5 — Repeated 3×10-fold cross-validation (tab:paired-tuned, tab:paired-full)
Corrected resampled-t over 30 folds (Bouckaert & Frank): same variance factor 1/J + n_test/n_train with J = 30, df = 29. "Single partition" = the same statistic from repeat 0 only (the main run). Holm and BH are computed within the 31 main and within the 14 MLTE comparisons.

In [ ]:
def rcv_pair(model, a, b, met):
    X = WR[(model, met)]
    d = (X[a] - X[b])
    m, lo, hi, p = corrected_t(d.values)
    m0, lo0, hi0, p0 = corrected_t(d.loc[0].values)
    return dict(delta=m, lo=lo, hi=hi, p=p, folds_ahead=int((d.values > 0).sum()), n=len(d), lo0=lo0, hi0=hi0, p0=p0,
                width_ratio=(hi - lo) / (hi0 - lo0), delta0=m0,
                sd_rep=d.groupby(level=0).mean().std(ddof=1), rng_rep=d.groupby(level=0).mean().max() - d.groupby(level=0).mean().min())
RC = {}
for model in (TLR, DLR):
    for met in ('auc', 'aupr'):
        fam = {}
        for name, pairs in (('main', MAIN_PAIRS), ('mlte', MLTE_RCV_PAIRS)):
            f = pd.DataFrame({(a, b): rcv_pair(model, a, b, met) for a, b in pairs}).T
            f['holm'] = holm(f.p.values); f['bh'] = bh(f.p.values); f['family'] = name
            fam[name] = f
        RC[(model, met)] = pd.concat(fam.values())

# validation against the precomputed rcv_*.csv files (the experiments notebook wrote them): max absolute differences
chk = []
for model, sfx in ((TLR, ''), (DLR, '_logisticregression')):
    for stem, key in (('main', 'main'), ('mlte', 'mlte')):
        f = read(f'rcv_{stem}_auroc{sfx}.csv', index_col=0)
        f.index = pd.MultiIndex.from_tuples([_split(c) for c in f.index])
        mine = RC[(model, 'auc')].loc[f.index]
        chk.append(dict(classifier=CLF_NAME[model], family=stem, comparisons=len(f),
                        max_abs_diff_delta=(mine.delta - f.delta).abs().max(), max_abs_diff_lo=(mine.lo - f.lo).abs().max(),
                        max_abs_diff_p=(mine.p - f.p).abs().max(), max_abs_diff_holm=(mine.holm - f.p_holm).abs().max(),
                        max_abs_diff_width_ratio=(mine.width_ratio - f.width_ratio_vs_rep0).abs().max(),
                        folds_ahead_equal=bool((mine.folds_ahead == f.folds_ahead).all())))
chk = pd.DataFrame(chk).set_index(['classifier', 'family'])
show('rcv_vs_files', chk, 'Recomputed repeated-CV statistics vs the stored rcv_*.csv files')

def RG(model, met, a, b):
    """stored repeated-CV row for (a, b), or recomputed on the fly when only the reverse / an unlisted pair is stored"""
    A = RC[(model, met)]
    if (a, b) in A.index: return A.loc[(a, b)]
    r = pd.Series(rcv_pair(model, a, b, met)); r['holm'] = np.nan; r['bh'] = np.nan
    if (b, a) in A.index: r['holm'] = A.loc[(b, a)].holm; r['bh'] = A.loc[(b, a)].bh
    return r
def rcv_table(model, MS):
    rows = {}
    for lbl, *_ in MS:
        a, b = pair(lbl)
        r = RG(model, 'auc', a, b); u = RG(model, 'aupr', a, b)
        rows[lbl] = {'AUROC Δ': r.delta, 'folds ahead': f'{r.folds_ahead}/{r.n}', 'repeated-CV interval': ci(r.lo, r.hi, 4),
                     'single partition': ci(r.lo0, r.hi0, 4), 'Holm p': r.holm, 'BH p': r.bh,
                     'AUPR Δ': u.delta, 'AUPR folds': f'{u.folds_ahead}/{u.n}', 'AUPR interval': ci(u.lo, u.hi, 4)}
    return pd.DataFrame(rows).T
t_tuned = rcv_table(TLR, MS_RCV_TUNED); t_def = rcv_table(DLR, MS_RCV_DEFAULT)
show('paired_tuned', t_tuned, 'tab:paired-tuned — tuned logistic regression (primary), 3×10-fold')
show('paired_full', t_def, 'tab:paired-full — default L2 logistic regression, 3×10-fold')
for model, MS, tag in ((TLR, MS_RCV_TUNED, 'paired-tuned'), (DLR, MS_RCV_DEFAULT, 'paired-full')):
    for (lbl, d, fo, lo, hi, lo0, hi0, ud, uf, ulo, uhi) in MS:
        a, b = pair(lbl); r = RG(model, 'auc', a, b); u = RG(model, 'aupr', a, b)
        S = f'tab:{tag}'
        claim(S, f'{lbl}: AUROC Δ', d, r.delta, 4, True); claim(S, f'{lbl}: folds ahead', fo, r.folds_ahead, 0)
        claim(S, f'{lbl}: RCV lower', lo, r.lo, 4, True); claim(S, f'{lbl}: RCV upper', hi, r.hi, 4, True)
        claim(S, f'{lbl}: single-partition lower', lo0, r.lo0, 4, True); claim(S, f'{lbl}: single-partition upper', hi0, r.hi0, 4, True)
        claim(S, f'{lbl}: AUPR Δ', ud, u.delta, 4, True); claim(S, f'{lbl}: AUPR folds ahead', uf, u.folds_ahead, 0)
        claim(S, f'{lbl}: AUPR lower', ulo, u.lo, 4, True); claim(S, f'{lbl}: AUPR upper', uhi, u.hi, 4, True)

# ---- paragraph statistics
arm_rep = RCV.groupby(['model', 'arm', 'repeat']).auc.mean().unstack('repeat')
arm_tab = pd.concat({CLF_NAME[m]: pd.DataFrame({'mean over 30 folds': RCV[RCV.model == m].groupby('arm').auc.mean(),
                                                 'SD of 3 repeat means': arm_rep.loc[m].std(axis=1, ddof=1),
                                                 'max-min over repeats': arm_rep.loc[m].max(axis=1) - arm_rep.loc[m].min(axis=1)}) for m in (TLR, DLR)}, axis=1)
arm_tab.index = [ARM2LABEL.get(a, a) for a in arm_tab.index]
show('rcv_arm_means', arm_tab, 'Mean AUROC over 30 folds and between-repeat spread, per representation')
rcva = read('rcv_arms.csv'); 
print('max |SD of repeat means - rcv_arms.csv| =', max(abs(arm_rep.loc[TLR].std(axis=1, ddof=1).reindex(rcva[rcva.metric == 'auc'].arm.values).values - rcva[rcva.metric == 'auc'].sd_of_repeat_means.values)))

for model, wr_ms, rg, mov, sdr, h31, hs, hm14 in ((TLR, .76, (.56, 1.11), .003, (.0001, .0016), 14, 10, 11), (DLR, .79, (.60, 1.16), .005, (.0003, .0026), 14, 8, 8)):
    A = RC[(model, 'auc')]; main = A[A.family == 'main']; ml = A[A.family == 'mlte']
    nm = CLF_NAME[model]
    claim('App. RCV', f'{nm}: mean width ratio repeated / single (31 main)', wr_ms, main.width_ratio.mean(), 2)
    claim_range('App. RCV', f'{nm}: range of width ratios', rg[0], rg[1], main.width_ratio, 2)
    main_arms = arm_rep.loc[model].loc[[a for a in arm_rep.loc[model].index if a in MAIN14 + ['mlte_50']]]
    sds = main_arms.std(axis=1, ddof=1); mv = main_arms.max(axis=1) - main_arms.min(axis=1)
    claim('App. RCV', f'{nm}: largest between-repeat move of an arm mean ("at most / up to")', mov, mv.max(), 3)
    claim_range('App. RCV', f'{nm}: SD of the three repeat means (per representation)', sdr[0], sdr[1], sds, 4,
                note=f'text says "of a given representation"; the same range taken over the 45 contrasts is {A.sd_rep.min():.4f} to {A.sd_rep.max():.4f}')
    claim_range('App. RCV', f'{nm}: SD of the three repeat means (per contrast)', sdr[0], sdr[1], A.sd_rep, 4, note='alternative reading: spread of the three repeat means of each contrast')
    claim('App. RCV', f'{nm}: main AUROC contrasts with Holm p<0.05 (repeated)', h31, (main.holm < .05).sum(), 0)
    claim('App. RCV', f'{nm}: main AUROC contrasts with Holm p<0.05 (single partition, corrected-t)', hs, (CONTR[model][CONTR[model].family == 'main'].ct_holm < .05).sum(), 0)
    claim('App. RCV', f'{nm}: MLTE contrasts with Holm p<0.05 (repeated)', hm14, (ml.holm < .05).sum(), 0)
    same = (np.sign(main.delta) == np.sign(main.delta0)).sum()
    print(nm, 'direction unchanged (RCV vs single partition):', same, 'of 31')
claim('App. RCV', 'default L2: direction unchanged for all but one of 31 main contrasts', 30,
      (np.sign(RC[(DLR, 'auc')].query("family=='main'").delta) == np.sign(RC[(DLR, 'auc')].query("family=='main'").delta0)).sum(), 0)
dl = RC[(DLR, 'auc')]
gc = pd.DataFrame({(g, c): rcv_pair(DLR, g, c, 'auc') for g in GRAPHS4 for c in CTRL3}).T
claim_range('App. RCV', 'default L2: graphs exceed controls in 29-30 of 30 folds', 29, 30, gc.loc[HAND3].folds_ahead, 0)
claim_range('App. RCV', 'default L2: handcrafted graph - control, 0.017 to 0.025 AUROC', .017, .025, gc.loc[HAND3].delta, 3, note=f'including the MLTE graph the range is up to {gc.delta.max():.3f}')
claim('App. RCV', 'default L2: MLTE graph ahead of controls in 30 of 30', 30, gc.loc['mlte_50'].folds_ahead.min(), 0)
gct = pd.DataFrame({(g, c): rcv_pair(TLR, g, c, 'auc') for g in GRAPHS4 for c in CTRL3}).T
claim_range('tab:answers Q1', 'tuned LR: graph ahead of controls in 29-30 of 30 folds', 29, 30, gct.folds_ahead, 0)
claim_range('tab:answers Q2', 'tuned LR repeated CV: graph+onset - onset', .0003, .0065, [RG(TLR, 'auc', og2[l], 'raw_onset').delta for l in og], 4)

arm_ms_t = {'Raw': .655, 'Raw + onset': .677, 'OAO': .668, 'MDBT': .674, 'DCEG': .671, 'MLTE': .678, 'Random graph': .654, 'Age-and-sex graph': .654,
            'Shuffled history (DCEG)': .655, 'OAO + onset': .677, 'MDBT + onset': .679, 'DCEG + onset': .679, 'MLTE + onset': .683, 'Raw + MLTE vector': .681}
arm_ms_d = {'Raw': .654, 'Raw + onset': .663, 'OAO': .667, 'MDBT': .672, 'DCEG': .667, 'MLTE': .677, 'Random graph': .648, 'Age-and-sex graph': .647,
            'Shuffled history (DCEG)': .650, 'Raw + MLTE vector': .674, 'MLTE + onset': .664}
for lbl, v in arm_ms_t.items(): claim('App. RCV', f'tuned LR mean AUROC over 30 folds, {lbl}', v, arm_tab.loc[lbl, (CLF_NAME[TLR], 'mean over 30 folds')], 3)
for lbl, v in arm_ms_d.items(): claim('App. RCV', f'default L2 mean AUROC over 30 folds, {lbl}', v, arm_tab.loc[lbl, (CLF_NAME[DLR], 'mean over 30 folds')], 3)
claim_range('App. RCV', 'default L2 combined 0.660-0.662 (OAO/MDBT/DCEG + onset)', .660, .662, [arm_tab.loc[l, (CLF_NAME[DLR], 'mean over 30 folds')] for l in ('OAO + onset', 'MDBT + onset', 'DCEG + onset')], 3)
claim('App. RCV', 'tuned LR six content-group counts', .682, arm_tab.loc['raw_countgroups', (CLF_NAME[TLR], 'mean over 30 folds')], 3)
claim('App. RCV', 'default L2 six content-group counts', .681, arm_tab.loc['raw_countgroups', (CLF_NAME[DLR], 'mean over 30 folds')], 3)

# Part 6 — The 26 additional arms, vocabulary ablation and sensitivity families

In [ ]:
ARMS26 = ['raw_presence', 'raw_count', 'raw_countgroups', 'raw_onset_m1', 'raw_onset__no_F41_F43', 'raw_onset__no_primary_care_type',
          'mdbt_50_noF41F43', 'dceg_50_noF41F43', 'oaopool500_50_noF41F43', 'oaopool500_50_noF41F43_onset', 'mdbt_50_noF41F43_onset', 'dceg_50_noF41F43_onset',
          'oao_50', 'wjpool500_50', 'wjpool500_50_presence', 'raw_mdbtvec', 'raw_dcegvec', 'mdbtvec_only', 'dcegvec_only', 'mdbt_50_mdbtvec', 'dceg_50_dcegvec',
          'random_50_mdbtvec', 'random_50_dcegvec', 'random_50_onset', 'featknn_50', 'shuf_dceg_50']
assert len(ARMS26) == len(MS_ARMS26) == 26
for a in ARMS26: assert a in set(PF.arm), a
ARM_LBL26 = dict(zip(ARMS26, [r[0] for r in MS_ARMS26]))
ARM2LABEL.update(ARM_LBL26)
nf = PF.groupby('arm').n_features.first()
rows = {}
for a in ARMS26:
    d = W[(TLR, 'auc')][a]; u = W[(TLR, 'aupr')][a]
    rows[ARM_LBL26[a]] = {'columns': nf[a], 'AUROC mean': d.mean(), 'AUROC SD': d.std(ddof=1), 'bootstrap lo': BA[TLR].loc[a, 'AUROC_lo'], 'bootstrap hi': BA[TLR].loc[a, 'AUROC_hi'],
                          'AUPR mean': u.mean(), 'AUPR SD': u.std(ddof=1), 'Default L2 AUROC': W[(DLR, 'auc')][a].mean(), 'Default CB AUROC': W[(DCB, 'auc')][a].mean()}
t26 = pd.DataFrame(rows).T
show('arms26', t26.round(4), 'tab:arms26 — the 26 additional arms (tuned LR mean ± SD, bootstrap interval; default L2 and default CatBoost means)')
for a, (lbl, cols, m, sd, blo, bhi, pm, psd, dl, dc) in zip(ARMS26, MS_ARMS26):
    r = t26.loc[ARM_LBL26[a]]
    claim('tab:arms26', f'{lbl}: columns', cols, r['columns'], 0)
    claim('tab:arms26', f'{lbl}: AUROC mean', m, r['AUROC mean'], 4); claim('tab:arms26', f'{lbl}: AUROC SD', sd, r['AUROC SD'], 4)
    claim('tab:arms26', f'{lbl}: bootstrap lower', blo, r['bootstrap lo'], 4); claim('tab:arms26', f'{lbl}: bootstrap upper', bhi, r['bootstrap hi'], 4)
    claim('tab:arms26', f'{lbl}: AUPR mean', pm, r['AUPR mean'], 4); claim('tab:arms26', f'{lbl}: AUPR SD', psd, r['AUPR SD'], 4)
    claim('tab:arms26', f'{lbl}: default L2 AUROC', dl, r['Default L2 AUROC'], 4); claim('tab:arms26', f'{lbl}: default CatBoost AUROC', dc, r['Default CB AUROC'], 4)
# duplicate arm: raw_onsetscaled must equal raw_onset
dup = {CLF_NAME[m]: float(abs(W[(m, 'auc')]['raw_onsetscaled'] - W[(m, 'auc')]['raw_onset']).max()) for m in CLF.values() if 'raw_onsetscaled' in W[(m, 'auc')].columns}
print('raw_onsetscaled is a duplicate label of raw_onset; max |difference| in fold AUROC by classifier (only classifiers that have the arm):', dup)

## F41/F43 ablation (tab:nof) and sensitivity family

In [ ]:
nof = {}
for m in (TLR, DLR):
    f = BOOT[('nof', m, 'auroc')]
    for i, (c, r) in enumerate(f.iterrows()):
        x, y = _split(c); nof.setdefault(pair_label(x, y) if x in ARM2LABEL and y in ARM2LABEL else c, {})
t_nof = pd.concat({CLF_NAME[m]: BOOT[('nof', m, 'auroc')][['delta', 'lo', 'hi', 'p_holm']] for m in (TLR, DLR)}, axis=1)
t_nof.index = [r[0] for r in MS_NOF]
show('nof', t_nof, 'tab:nof — F41/F43 ablation contrasts (bootstrap, Holm within the 19-comparison family)')
assert len(BOOT[('nof', TLR, 'auroc')]) == len(MS_NOF) == 19
for i, (lbl, d1, l1, h1, p1, d2, l2, h2, p2) in enumerate(MS_NOF):
    for m, (d, l, h, p) in ((TLR, (d1, l1, h1, p1)), (DLR, (d2, l2, h2, p2))):
        r = BOOT[('nof', m, 'auroc')].iloc[i]
        claim('tab:nof', f'{lbl}: {CLF_NAME[m]} Δ', d, r.delta, 4, True); claim('tab:nof', f'{lbl}: {CLF_NAME[m]} lower', l, r.lo, 4, True); claim('tab:nof', f'{lbl}: {CLF_NAME[m]} upper', h, r.hi, 4, True)
        if p < 0: claim_true('tab:nof', f'{lbl}: {CLF_NAME[m]} Holm p printed as <{-p}', r.p_holm < -p, f'p = {r.p_holm:.5f}')
        else: claim('tab:nof', f'{lbl}: {CLF_NAME[m]} Holm p', p, r.p_holm, 3)

sens = pd.concat({CLF_NAME[m]: BOOT[('sens', m, 'auroc')][['delta', 'lo', 'hi', 'p_holm']] for m in (TLR, DLR, DCB) if ('sens', m, 'auroc') in BOOT}, axis=1)
sens.index = ['OAO untimed (WJ pool 500) vs OAO', 'counts vs presence-only', 'counts vs single count', 'no-F41/F43 onset vs no-primary-care-type onset']
show('sens_family', sens, 'The four sensitivity-family comparisons (bootstrap, Holm within the family of 4)')
S = BOOT[('sens', TLR, 'auroc')]; SD_ = BOOT[('sens', DLR, 'auroc')]
claim('App. sensitivity', 'counts vs single count Δ (tuned LR)', .0062, S.iloc[2].delta, 4, True); claim('App. sensitivity', '  lower', .0039, S.iloc[2].lo, 4, True); claim('App. sensitivity', '  upper', .0086, S.iloc[2].hi, 4, True)
claim_true('App. sensitivity', '  Holm p < 0.001', S.iloc[2].p_holm < .001, f'{S.iloc[2].p_holm:.2e}')
claim('App. sensitivity', 'F41/F43 vs primary-care-type removal Δ', -.0058, S.iloc[3].delta, 4, True); claim('App. sensitivity', '  lower', -.0087, S.iloc[3].lo, 4, True); claim('App. sensitivity', '  upper', -.0029, S.iloc[3].hi, 4, True)
claim_true('App. sensitivity', '  Holm p < 0.001', S.iloc[3].p_holm < .001, f'{S.iloc[3].p_holm:.2e}')
claim('App. sensitivity', 'counts vs presence-only Δ', .0030, S.iloc[1].delta, 4, True); claim('App. sensitivity', '  lower', .0006, S.iloc[1].lo, 4, True); claim('App. sensitivity', '  upper', .0059, S.iloc[1].hi, 4, True)
claim('App. sensitivity', '  Holm p', .054, S.iloc[1].p_holm, 3)
claim('App. sensitivity', 'untimed vs timed OAO Δ', -.0005, S.iloc[0].delta, 4, True); claim('App. sensitivity', '  lower', -.0043, S.iloc[0].lo, 4, True); claim('App. sensitivity', '  upper', .0034, S.iloc[0].hi, 4, True)
claim('App. sensitivity', '  p', .81, S.iloc[0].p_holm, 2)
claim('App. sensitivity', 'default L2 counts vs presence-only Δ', .0116, SD_.iloc[1].delta, 4, True); claim('App. sensitivity', '  lower', .0075, SD_.iloc[1].lo, 4, True); claim('App. sensitivity', '  upper', .0160, SD_.iloc[1].hi, 4, True)
claim('App. sensitivity', 'default L2 counts vs single count Δ', .0067, SD_.iloc[2].delta, 4, True); claim('App. sensitivity', '  lower', .0041, SD_.iloc[2].lo, 4, True); claim('App. sensitivity', '  upper', .0092, SD_.iloc[2].hi, 4, True)
claim('App. sensitivity', 'default L2 F41/F43 vs primary-care-type removal Δ', -.0084, SD_.iloc[3].delta, 4, True); claim('App. sensitivity', '  lower', -.0118, SD_.iloc[3].lo, 4, True); claim('App. sensitivity', '  upper', -.0049, SD_.iloc[3].hi, 4, True)

## Numbers quoted in the "Findings" bullets of the sensitivity appendix
Each line is a mean AUROC of an arm or a paired difference (corrected-t interval, from the ten folds of the main run unless marked repeated CV).

In [ ]:
mean_rows = [  # (statement, model, arm, manuscript value)
 ('Counts', TLR, 'raw_countgroups', .6819), ('Counts', DLR, 'raw_countgroups', .6815), ('Counts', DCB, 'raw_countgroups', .6788),
 ('Raw + MDBT descriptor', TLR, 'raw_mdbtvec', .6827), ('Raw + MDBT descriptor', DLR, 'raw_mdbtvec', .6797), ('Raw + MDBT descriptor', DCB, 'raw_mdbtvec', .6811),
 ('MDBT graph', TLR, 'mdbt_50', .6742), ('MDBT graph', DLR, 'mdbt_50', .6717), ('MDBT graph', DCB, 'mdbt_50', .6684),
 ('MDBT + MDBT descriptor', TLR, 'mdbt_50_mdbtvec', .6828), ('Random + MDBT descriptor', TLR, 'random_50_mdbtvec', .6827),
 ('MDBT + MDBT descriptor', DLR, 'mdbt_50_mdbtvec', .6756), ('Random + MDBT descriptor', DLR, 'random_50_mdbtvec', .6746),
 ('MDBT + MDBT descriptor', DCB, 'mdbt_50_mdbtvec', .6770), ('Random + MDBT descriptor', DCB, 'random_50_mdbtvec', .6778),
 ('Feature-kNN', TLR, 'featknn_50', .6537), ('Feature-kNN', DLR, 'featknn_50', .6492), ('Feature-kNN', DCB, 'featknn_50', .6397),
 ('Node-shuffled DCEG', TLR, 'shuf_dceg_50', .6547), ('Random graph + onset', TLR, 'random_50_onset', .6782),
 ('-1 onset coding', TLR, 'raw_onset_m1', .6700), ('standardized onset coding', TLR, 'raw_onset', .6786),
 ('-1 onset coding', DLR, 'raw_onset_m1', .6677), ('standardized onset coding', DLR, 'raw_onset', .6641),
 ('Presence indicators alone', TLR, 'raw_presence', .6789), ('Presence indicators alone', DLR, 'raw_presence', .6699), ('Presence indicators alone', DCB, 'raw_presence', .6780)]
for s_, m, a, v in mean_rows: claim('App. sensitivity (bullets)', f'{s_}: mean AUROC, {CLF_NAME[m]}', v, W[(m, 'auc')][a].mean(), 4)

diff_rows = [  # (statement, model, a, b, delta, ci lo, ci hi, decimals of the interval)
 ('counts - onset columns (468)', TLR, 'raw_countgroups', 'raw_onset', .0034, .0005, .0063, 4),
 ('counts - onset columns (468)', DLR, 'raw_countgroups', 'raw_onset', .0174, .0095, .0253, 4),
 ('counts - onset columns (468)', DCB, 'raw_countgroups', 'raw_onset', -.0003, -.016, .016, 3),
 ('OAO 500 pool - 51 pool', TLR, 'oaopool500_50', 'oao_50', .0045, -.004, .013, 3), ('OAO 500 pool - 51 pool', DLR, 'oaopool500_50', 'oao_50', .0047, None, None, 3),
 ('OAO 500 pool - 51 pool', DCB, 'oaopool500_50', 'oao_50', -.0028, None, None, 3),
 ('untimed - timed OAO', DLR, 'wjpool500_50', 'oaopool500_50', -.0013, None, None, 3),
 ('onset without F41/F43 - raw', TLR, 'raw_onset__no_F41_F43', 'raw', .0178, .011, .025, 3), ('onset without F41/F43 - raw', DCB, 'raw_onset__no_F41_F43', 'raw', .0188, .008, .030, 3),
 ('onset without F41/F43 - raw', DLR, 'raw_onset__no_F41_F43', 'raw', .0031, -.005, .011, 3),
 ('loss vs full onset columns', TLR, 'raw_onset', 'raw_onset__no_F41_F43', .0057, None, None, 3), ('loss vs full onset columns', DCB, 'raw_onset', 'raw_onset__no_F41_F43', .0103, None, None, 3),
 ('loss vs full onset columns', DLR, 'raw_onset', 'raw_onset__no_F41_F43', .0062, None, None, 3),
 ('raw + MDBT descriptor - MDBT graph', TLR, 'raw_mdbtvec', 'mdbt_50', .0084, .0015, .0154, 4),
 ('DCEG random graph + embedding lower than DCEG + embedding', TLR, 'dceg_50_dcegvec', 'random_50_dcegvec', .0048, .0003, .0093, 4),
 ('DCEG + embedding - random + embedding', DLR, 'dceg_50_dcegvec', 'random_50_dcegvec', .0079, None, None, 3), ('DCEG + embedding - random + embedding', DCB, 'dceg_50_dcegvec', 'random_50_dcegvec', .0039, None, None, 3),
 ('feature-kNN - random graph', TLR, 'featknn_50', 'random_50', -.0009, -.003, .001, 3),
 ('feature-kNN - handcrafted (min)', TLR, 'mdbt_50', 'featknn_50', None, None, None, 3),
 ('node-shuffled - shufrow DCEG', TLR, 'shuf_dceg_50', 'shufrow_dceg_50', -.0006, -.002, .001, 3),
 ('random + onset - onset', TLR, 'random_50_onset', 'raw_onset', -.0003, None, None, 3),
 ('-1 coding vs standardized (standardized better by)', TLR, 'raw_onset', 'raw_onset_m1', .0086, .0025, .0146, 4),
 ('-1 coding vs standardized (standardized worse by)', DLR, 'raw_onset', 'raw_onset_m1', -.0036, None, None, 3),
 ('-1 coding vs standardized', DCB, 'raw_onset', 'raw_onset_m1', .0023, None, None, 3),
 ('presence - 468 columns', TLR, 'raw_presence', 'raw_onset', .0003, None, None, 3), ('presence - 468 columns', DLR, 'raw_presence', 'raw_onset', .0058, .0011, .0105, 4),
 ('presence - 468 columns', DCB, 'raw_presence', 'raw_onset', -.0012, None, None, 3),
 ('no primary-care-type - onset', TLR, 'raw_onset__no_primary_care_type', 'raw_onset', -.0001, None, None, 3)]  # the text prints -0.0001; the files give +0.0001
out = []
for s_, m, a, b, d, lo, hi, dd in diff_rows:
    r = pair_row(m, a, b)
    out.append({'statement': s_, 'classifier': CLF_NAME[m], 'a - b': pair_label(a, b), 'Δ': r['delta'], 'corrected-t interval': ci(r['ct_lo'], r['ct_hi'], 4)})
    if d is not None: claim('App. sensitivity (bullets)', f'{s_} ({CLF_NAME[m]}): Δ', d, r['delta'], 4, True)
    if lo is not None: claim('App. sensitivity (bullets)', f'{s_} ({CLF_NAME[m]}): lower', lo, r['ct_lo'], dd, True); claim('App. sensitivity (bullets)', f'{s_} ({CLF_NAME[m]}): upper', hi, r['ct_hi'], dd, True)
show('sens_bullets', pd.DataFrame(out).set_index('statement'), 'Paired differences quoted in the sensitivity findings (corrected-t, ten folds of the main run)')

# repeated-CV numbers quoted in the bullets
for s_, m, a, b, d, lo, hi in (('counts - onset (repeated CV)', TLR, 'raw_countgroups', 'raw_onset', .0046, .0015, .0077), ('counts - onset (repeated CV)', DLR, 'raw_countgroups', 'raw_onset', .0182, .0124, .0240),
                               ('onset without F41/F43 - raw (repeated CV)', TLR, 'raw_onset__no_F41_F43', 'raw', .0166, .0114, .0217), ('onset without F41/F43 - raw (repeated CV)', DLR, 'raw_onset__no_F41_F43', 'raw', .0017, -.006, .010)):
    r = RG(m, 'auc', a, b)
    dd = 3 if (m == DLR and a == 'raw_onset__no_F41_F43') else 4
    claim('App. sensitivity (bullets)', f'{s_} {CLF_NAME[m]}: Δ', d, r.delta, 4, True)
    claim('App. sensitivity (bullets)', f'  lower', lo, r.lo, dd, True); claim('App. sensitivity (bullets)', f'  upper', hi, r.hi, dd, True)
claim('App. sensitivity (bullets)', 'loss of F41/F43 onset vs full, repeated CV, tuned LR', .0056, -RG(TLR, 'auc', 'raw_onset__no_F41_F43', 'raw_onset').delta, 4)
claim('App. sensitivity (bullets)', 'loss of F41/F43 onset vs full, repeated CV, default L2', .0067, -RG(DLR, 'auc', 'raw_onset__no_F41_F43', 'raw_onset').delta, 4)

## Graph tie diagnostics (tab:graphdiag)
These come from the graph-construction logs and are not in any result file; they are shown as typed in.

In [ ]:
gd = pd.DataFrame.from_dict(STATIC['graphdiag'], orient='index', columns=['boundary-tie share', 'lo', 'hi'])
show('graphdiag', gd, 'tab:graphdiag — boundary-tie share (typed in)')
claim_na('tab:graphdiag', 'boundary-tie shares of the graph searches', '9 rows', note='from graph-construction logs, not in the result files')

# Part 7 — Numbers quoted in the Abstract, Results, Discussion and Conclusions
Everything that is a statement of effect size in the running text, recomputed from the contrast tables above.

In [ ]:
d = lambda m, a, b: C(m, a, b).delta
GR_RAW = [d(TLR, g, 'raw') for g in GRAPHS4]
GR_CTL = [d(TLR, g, c) for g in HAND3 + ['mlte_50'] for c in CTRL3 if (g, c) in CONTR[TLR].index or (c, g) in CONTR[TLR].index]
claim_range('Abstract / Results Q1', 'every graph improved on raw features by 0.013-0.023 (tuned LR)', .013, .023, GR_RAW, 3)
claim_range('Results Q1', 'graphs vs control graphs 0.013-0.023 (tuned LR; all pairs available)', .013, .023, GR_CTL, 3, note=f'{len(GR_CTL)} graph-control pairs found')
claim_true('Results Q1', 'all corrected-t intervals for graph-vs-raw and graph-vs-control exclude zero',
           all(C(TLR, g, 'raw').ct_lo > 0 for g in GRAPHS4) and all((C(TLR, g, c).ct_lo > 0) for g in HAND3 + ['mlte_50'] for c in CTRL3 if (g, c) in CONTR[TLR].index or (c, g) in CONTR[TLR].index),
           'min lower limit = %.4f' % min([C(TLR, g, 'raw').ct_lo for g in GRAPHS4] + [C(TLR, g, c).ct_lo for g in HAND3 + ['mlte_50'] for c in CTRL3 if (g, c) in CONTR[TLR].index or (c, g) in CONTR[TLR].index]))
claim_range('Results Q1', 'controls fell 0.003-0.008 below raw (default L2 and tuned CatBoost)', -.008, -.003, [d(m, c, 'raw') for m in (DLR, TCB) for c in CTRL3], 3, True)
claim_range('Discussion', 'default L2: controls 0.005-0.008 below raw', -.008, -.005, [d(DLR, c, 'raw') for c in CTRL3], 3, True)
claim('Abstract', 'onset-age columns improved on raw features by 0.023', .023, d(TLR, 'raw_onset', 'raw'), 3)
claim('Results Q2', 'six count features reached 0.682 (tuned LR)', .682, W[(TLR, 'auc')]['raw_countgroups'].mean(), 3)
claim('Results Q2', 'counts 0.003 above onset columns', .003, d(TLR, 'raw_countgroups', 'raw_onset'), 3)
claim_range('Abstract', 'adding graph-propagated features to onset columns gained at most 0.005; range +0.0002..+0.0054', .0002, .0054, [d(TLR, og2[l], 'raw_onset') for l in og], 4, True)
claim('Results Q2', 'upper limit for OAO + onset vs onset under tuned CatBoost (+0.011)', .011, C(TCB, 'oaopool500_50_onset', 'raw_onset').ct_hi, 3, True)
claim_range('Results Q2', 'adding onset ages to a graph helped by +0.006 to +0.010 (tuned LR)', .006, .010, [d(TLR, og2[l], og[l]) for l in og], 3, True)
claim_true('Results Q2', 'every upper limit of graph+onset vs onset is below the 0.01 margin (tuned LR)', max(C(TLR, og2[l], 'raw_onset').ct_hi for l in og) < .01, f'max upper = {max(C(TLR, og2[l], "raw_onset").ct_hi for l in og):.4f}')
claim_true('Results Q2', 'the largest gain (MLTE + onset) does not reproduce under the other classifiers (|Δ| <= 0.0011)',
           max(abs(d(DLR, 'mlte_50_onset', 'raw_onset')), abs(d(TCB, 'mlte_50_onset', 'raw_onset'))) < .0012,
           f'default L2 {d(DLR, "mlte_50_onset", "raw_onset"):+.4f}, tuned CB {d(TCB, "mlte_50_onset", "raw_onset"):+.4f}')

claim('Results Q4', 'MLTE graph AUROC 0.678', .678, W[(TLR, 'auc')]['mlte_50'].mean(), 3)
claim_range('Results Q4', 'MLTE above OAO and DCEG by 0.006-0.009', .006, .009, [d(TLR, 'mlte_50', 'oaopool500_50'), d(TLR, 'mlte_50', 'dceg_50')], 3, True)
_r = C(TLR, 'mlte_50', 'mdbt_50')
claim_true('Results Q4', 'MLTE level with MDBT: interval includes zero and |Δ| < 0.005', (_r.ct_lo < 0 < _r.ct_hi) and abs(_r.delta) < .005, f'Δ = {_r.delta:+.4f}, corrected-t [{_r.ct_lo:+.4f}, {_r.ct_hi:+.4f}]')
claim('Results Q4', 'raw + MLTE vector 0.682', .682, W[(TLR, 'auc')]['raw_mltevec'].mean(), 3)
claim('Results Q4', 'raw + MLTE vector - MLTE', .004, d(TLR, 'raw_mltevec', 'mlte_50'), 3, True)
claim('Results Q4', 'MLTE + onset - onset', .005, d(TLR, 'mlte_50_onset', 'raw_onset'), 3, True)
claim('Results Q4', '  lower (corrected-t)', .001, C(TLR, 'mlte_50_onset', 'raw_onset').ct_lo, 3, True); claim('Results Q4', '  upper', .009, C(TLR, 'mlte_50_onset', 'raw_onset').ct_hi, 3, True)
claim('Results Q4', 'MLTE + onset - onset under default L2', .001, d(DLR, 'mlte_50_onset', 'raw_onset'), 3, True)
claim('Results Q4', 'MLTE + onset - onset under tuned CatBoost', .0007, d(TCB, 'mlte_50_onset', 'raw_onset'), 4, True)
claim('Results Q4', 'onset columns 0.679', .679, W[(TLR, 'auc')]['raw_onset'].mean(), 3)

claim('Results Q3', 'default L2: onset ages gained 0.009 over raw', .009, d(DLR, 'raw_onset', 'raw'), 3)
claim_range('Results Q3', 'default L2: adding onset ages to a graph lowered AUROC by 0.005-0.012', -.012, -.005, [d(DLR, og2[l], og[l]) for l in og], 3, True)
claim('Results Q3', 'calibration slope of the 468-column default model', .73, cal_dlr.loc['Raw + onset', 'Slope'], 2)
claim('Results Q3', 'median C 0.10 (default C = 1)', .10, sel.iloc[0, 2], 2)
claim_range('Results Q3', 'tuning raised onset-containing models by 0.015-0.019 (five onset-containing representations; bootstrap Δ)', .015, .019,
            d_.loc[['raw_onset', 'oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset', 'mlte_50_onset'], 'delta'], 3, True, note='same quantity as in tab:answers Q3')
claim_true('Results Q3', 'graphs OAO, MDBT, MLTE essentially unchanged (<= 0.003)', d_.loc[['oaopool500_50', 'mdbt_50', 'mlte_50'], 'delta'].max() <= .0035, f'max = {d_.loc[["oaopool500_50", "mdbt_50", "mlte_50"], "delta"].max():.4f}')
claim_range('Results Q3', 'tuning the booster raised all 14 representations by 0.003-0.009', .003, .009, CCD['catboosttuned_vs_catboost']['auroc'].loc[MAIN14, 'delta'], 3, True)

# sensitivity paragraph in Results
A_ = RC[(TLR, 'auc')]; Am = A_[A_.family == 'main']
same_sign = int((np.sign(Am.delta) == np.sign(Am.delta0)).sum())
claim('Results sensitivity', 'repeated CV reproduced the sign of this many of the 31 main contrasts (tuned LR)', 29, same_sign, 0)
flip = Am[np.sign(Am.delta) != np.sign(Am.delta0)]
print('contrasts whose sign differs between repeated CV and the single partition:'); print(pd.DataFrame({'repeated-CV Δ': flip.delta, 'single-partition Δ': flip.delta0}).rename(index=lambda x: x, level=None).to_string())
claim('Results sensitivity', 'the two exceptions were differences of +-0.0002 (largest |Δ| of the two, single partition)', .0002, flip.delta0.abs().max(), 4)
claim('Results sensitivity', 'removing F41/F43 lowered the onset gain by 0.006', .006, d(TLR, 'raw_onset', 'raw_onset__no_F41_F43'), 3)
_q = 100 * d(TLR, 'raw_onset', 'raw_onset__no_F41_F43') / d(TLR, 'raw_onset', 'raw')
claim_true('Abstract / Results', '"about a quarter" of the onset-column gain is lost without F41/F43 (share between 20% and 30%)', 20 <= _q <= 30, f'{_q:.1f}%')
claim('Results sensitivity', 'graph AUROC lowered by at most 0.004 after removing F41/F43', .004, max(-d(TLR, a, b) for a, b in (('oaopool500_50_noF41F43', 'oaopool500_50'), ('mdbt_50_noF41F43', 'mdbt_50'), ('dceg_50_noF41F43', 'dceg_50'))), 3)
claim('Results sensitivity', 'standardized onset ages beat -1 coding by 0.009 (tuned LR)', .009, d(TLR, 'raw_onset', 'raw_onset_m1'), 3)

# calibration paragraph
sl = pd.concat([PF[PF.model == m].groupby('arm').calibration_slope.mean().loc[MAIN14].rename(m) for m in (TLR, DLR, TCB)], axis=1)
sl_cal = pd.concat([PF[PF.model == m].groupby('arm').calibration_slope_cal.mean().loc[MAIN14].rename(m) for m in (TLR, DLR, TCB)], axis=1)
br_cal = pd.concat([PF[PF.model == m].groupby('arm').brier_cal.mean().loc[MAIN14].rename(m) for m in (TLR, DLR, TCB)], axis=1)
claim_range('Results calibration', 'tuned LR slopes 0.95-1.02', .95, 1.02, sl[TLR], 2)
claim_range('Results calibration', 'default L2 slopes, onset-containing representations 0.70-0.73', .70, .73, sl.loc[['raw_onset', 'oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset', 'mlte_50_onset'], DLR], 2)
claim('Results calibration', 'default L2 slope raw + MLTE vector', .81, sl.loc['raw_mltevec', DLR], 2)
claim_range('Results calibration', 'tuned CatBoost slopes 1.09-1.20', 1.09, 1.20, sl[TCB], 2)
claim_range('Results calibration', 'recalibrated slopes 0.99-1.06 (all three classifiers, 14 arms)', .99, 1.06, sl_cal.values.ravel(), 2)
claim_range('Results calibration', 'recalibrated Brier 0.0444-0.0450', .0444, .0450, br_cal.values.ravel(), 4)
claim('Results calibration', 'constant-prediction Brier 0.0457', .0457, rate * (1 - rate), 4)

# Discussion
claim('Discussion', 'MDBT graph: share tied at the k-th neighbour (47%)', 47, 100 * gd.loc['MDBT', 'boundary-tie share'], 0, note='consistency with tab:graphdiag only (typed in; from the graph logs)')
claim('Discussion', 'DCEG graph: share tied at the k-th neighbour (27%)', 27, 100 * gd.loc['DCEG', 'boundary-tie share'], 0, note='consistency with tab:graphdiag only (typed in; from the graph logs)')

claim('Discussion', 'the added value of MLTE to onset columns (0.005)', .005, d(TLR, 'mlte_50_onset', 'raw_onset'), 3)

# Part 8 — Register of every recomputed claim
One row per number checked against the manuscript. **OK** = the recomputed value, rounded half-up to the printed precision, equals the printed one. **CLOSE** = differs by one unit in the last printed digit (typically rounding of an already rounded value, or the 4-decimal bootstrap CSVs). **DIFF** = differs by more. **not recomputable** = needs data that is not among the supplied files.

In [ ]:
REG = pd.DataFrame(CLAIMS)
REG['status'] = pd.Categorical(REG.status, ['DIFF', 'CLOSE', 'not recomputable', 'OK'], ordered=True)
summ = REG.groupby(['section', 'status'], observed=True).size().unstack(fill_value=0)
summ['total'] = summ.sum(axis=1)
tot = summ.sum().to_frame('ALL').T
show('claims_summary', pd.concat([summ, tot]), 'Claims checked, by section and status')
show('claims_attention', REG[REG.status.isin(['DIFF', 'CLOSE'])].sort_values(['status', 'section'])[['status', 'section', 'statement', 'manuscript', 'recomputed', 'exact', 'note']].reset_index(drop=True),
     'Claims that do not reproduce exactly (DIFF first, then CLOSE)', index=False)
show('claims_not_recomputable', REG[REG.status == 'not recomputable'][['section', 'statement', 'manuscript', 'note']].reset_index(drop=True), 'Not recomputable from the supplied files', index=False)
show('claims_all', REG.sort_values(['section']).reset_index(drop=True), 'All claims (filter / sort in the notebook as needed)', index=False)

# Optional export
By default the notebook writes nothing. Set `WRITE_FILES = True` in the configuration cell to save every displayed table (and the claims register) into one Excel workbook at `EXPORT_PATH`.

In [ ]:
if WRITE_FILES:
    with pd.ExcelWriter(EXPORT_PATH) as xw:
        for name, df_ in TABLES.items():
            df_.to_excel(xw, sheet_name=name[:31])
    print('written:', EXPORT_PATH, '-', len(TABLES), 'sheets')
else:
    print('WRITE_FILES = False: no file written;', len(TABLES), 'tables are available in the dict TABLES')

# Figure `fig:forest` — principal paired contrasts (forest plot)
Draws the 27 principal paired AUROC contrasts (first minus second representation) of `tab:keycontrasts`: filled dots = primary tuned logistic regression with 95% corrected resampled-*t* intervals (thick bars); thin bars = patient-level bootstrap intervals; open markers = tuned CatBoost and default L2 logistic regression. The dashed lines mark the ±0.01 AUROC practical-relevance margin (interpretive, not pre-registered). Values are taken from the recomputed contrasts (`C(...)`) when available and otherwise from the printed `MS_KEY` values. The figure is only displayed; files are written only if `WRITE_FILES = True`.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

def _forest_rows():
    rows = []
    for (lbl, d0, cl0, ch0, bl0, bh0, dd0, dt0) in MS_KEY:          # printed values = fallback
        try:                                                         # recomputed values when the result files are loaded
            a, b = pair(lbl)
            r, rd, rt = C(TLR, a, b), C(DLR, a, b), C(TCB, a, b)
            rows.append(dict(label=lbl, d=r.delta, cl=r.ct_lo, ch=r.ct_hi, bl=r.boot_lo, bh=r.boot_hi, dlr=rd.delta, tcb=rt.delta))
        except Exception:
            rows.append(dict(label=lbl, d=d0, cl=cl0, ch=ch0, bl=bl0, bh=bh0, dlr=dd0, tcb=dt0))
    return pd.DataFrame(rows)

FOREST = _forest_rows()
MARGIN = 0.01

def draw_forest(df=FOREST):
    n = len(df)
    fig, ax = plt.subplots(figsize=(7.4, 0.27 * n + 1.4))
    y = np.arange(n)[::-1]
    ax.axvspan(-MARGIN, MARGIN, color='0.93', zorder=0)
    ax.axvline(0, color='0.35', lw=0.8, zorder=1)
    for x in (-MARGIN, MARGIN):
        ax.axvline(x, color='0.6', lw=0.7, ls='--', zorder=1)
    for yi, r in zip(y, df.itertuples()):
        ax.plot([r.cl, r.ch], [yi, yi], color='#1f4e79', lw=2.6, solid_capstyle='butt', zorder=2)
        if not (pd.isna(r.bl) or pd.isna(r.bh)):
            ax.plot([r.bl, r.bh], [yi, yi], color='#d97706', lw=1.0, zorder=3)
    ax.scatter(df.d, y, s=30, color='#1f4e79', zorder=4, label='Tuned LR (primary), corrected-$t$ 95% CI')
    ax.scatter(df.tcb, y, s=34, facecolors='none', edgecolors='#2a9d8f', marker='s', lw=1.2, zorder=4, label='Tuned CatBoost')
    ax.scatter(df.dlr, y, s=38, facecolors='none', edgecolors='#7b2d8e', marker='^', lw=1.2, zorder=4, label='Default L2 LR')
    ax.plot([], [], color='#d97706', lw=1.0, label='Patient-level bootstrap 95% CI (tuned LR)')
    ax.set_yticks(y); ax.set_yticklabels(df.label, fontsize=8)
    ax.set_xlabel('Difference in AUROC (first minus second representation)')
    ax.set_ylim(-0.8, n - 0.2)
    ax.grid(axis='x', color='0.85', lw=0.5, zorder=0); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.35, -0.07), ncol=2, fontsize=7.5, frameon=False)
    fig.tight_layout()
    return fig

fig_forest = draw_forest()
plt.show()
if WRITE_FILES:
    _base = os.path.splitext(EXPORT_PATH)[0]
    for ext in ('pdf', 'png'):
        fig_forest.savefig(f'{_base}_fig_forest.{ext}', dpi=300, bbox_inches='tight')
    print(f'saved {_base}_fig_forest.pdf / .png  ->  use in the manuscript via \\includegraphics for fig:forest')
else:
    print('WRITE_FILES = False: figure displayed only, nothing saved.')